# Phase 3 -- Visual Engine, Version 1: frozen CLIP + a small head

Implements `docs/PHASE_3_VISUAL_ENGINE.md` with the split design of
`docs/HANDBOOK_ADDENDUM_01_ROTATING_HOLDOUT.md`. Generated from `notebooks/src/05_visual_v1.py` by
`scripts/build_notebooks.py` -- edit the source, not this .ipynb.

**This notebook runs in stages, and stops after the stage named in `STAGE`** (first code cell):

| Stage | What it does | Needs |
|---|---|---|
| `task0` | Pre-flight: what is really attached, real OpenFake counts read from metadata only, a timed streamed sample, CLIP check, disk budget. **Default. Nothing else runs.** | -- |
| `scan` | Tasks 1-2, pass 1: one metadata row per image (no encoder) | the `PLAN` decided from Task 0 |
| `split` | Tasks 2-4: dedupe, group-aware splits with rotating held-out families, leakage checks, shortcut baselines | `scan` output |
| `extract` | Task 5: CLIP embeddings, resumable chunks | `split` output, shortcut gate |
| `heads` | Tasks 6-9: heads, calibration, evaluation suites | `extract` output |

**Kaggle settings:** GPU, Internet on, the `HF_TOKEN` secret attached (Add-ons -> Secrets), the
ArtiFact and FFHQ-thumbnails datasets added as inputs, and the notebook kept **private** (later
stages write per-image metadata and embeddings to its output, which the datasets' terms do not
allow in a public place). Go straight to **Save Version -> Save & Run All (Commit)**.

Every number printed here is from this run. Anything not run says "not yet run".

In [ ]:
import os
import subprocess
import sys

# True only when tests/smoke_notebook_05.py runs this notebook's code on synthetic data.
SMOKE_TEST = os.environ.get("TRUSTMESH_SMOKE") == "1"

# ---- THE STAGE SWITCH -------------------------------------------------------------------------------
# The last stage this run executes: task0 -> scan -> split -> extract -> heads.
# It stays "task0" until the Task 0 results have been read and the PLAN below has been decided.
STAGE = os.environ.get("TRUSTMESH_VISUAL_STAGE") or "task0"


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if not SMOKE_TEST:
    for module_name, package in [("pyarrow", "pyarrow"), ("huggingface_hub", "huggingface_hub"),
                                 ("transformers", "transformers"), ("fsspec", "fsspec"), ("xgboost", "xgboost")]:
        try:
            __import__(module_name)
        except ImportError:
            pip_install(package)

In [ ]:
import glob
import json
import shutil
import time
import traceback
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch

RUN_START = time.time()
SEED = 42
SEEDS = [42, 43, 44]
torch.manual_seed(SEED)
np.random.seed(SEED)

STAGES = ("task0", "scan", "split", "extract", "heads")
if STAGE not in STAGES:
    raise ValueError(f"STAGE must be one of {STAGES}, got {STAGE!r}")

OUTPUT_DIR = Path(os.environ.get("TRUSTMESH_OUTPUT_DIR", "./artifacts/visual"))
# Everything per-image (metadata rows, embeddings, predictions) and the trained weights go here.
# This folder is in .gitignore and must never reach a public repository or a public Kaggle dataset.
PRIVATE_DIR = OUTPUT_DIR / "private"
for folder in (OUTPUT_DIR, PRIVATE_DIR):
    folder.mkdir(parents=True, exist_ok=True)
# Output of an earlier session of this notebook, attached as an input, so finished chunks are reused.
RESUME_DIRS = [d for d in os.environ.get("TRUSTMESH_RESUME_DIRS", "").split(os.pathsep) if d] or \
    sorted(glob.glob("/kaggle/input/*/artifacts/visual/private"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE_TEST else "cpu")

N_AUG_VIEWS = 2                      # augmented views cached per training image (handbook: 2-3)
N_BOOTSTRAP = 30 if SMOKE_TEST else 1000
EMBED_BATCH = 8 if SMOKE_TEST else 64
THREADS = 2 if SMOKE_TEST else 4     # image decode threads (Kaggle gives 4 CPU cores)
SCAN_THREADS = 2 if SMOKE_TEST else 32
TARGET_RECALL, BAND_RECALL, BAND_FPR = 0.95, 0.99, 0.01   # operating point settings (KYC: favour recall)
TIME_BUDGET_HOURS = 9.0              # Kaggle stops a session at 12 h; stop cleanly well before
SHOW_DUPLICATE_PAIRS = SMOKE_TEST    # draws real images into the notebook output: private notebooks only
ACCEPT_RESIDUAL_SHORTCUT = SMOKE_TEST  # set True only after reading the shortcut-baseline numbers
# Heavier pixel baselines (Task 9): off by default, run only after Tasks 1-8 have real results.
RUN_SMALL_CNN = SMOKE_TEST or os.environ.get("TRUSTMESH_RUN_SMALL_CNN") == "1"
RUN_RESNET50 = SMOKE_TEST or os.environ.get("TRUSTMESH_RUN_RESNET50") == "1"

# ---- THE PLAN: decided from the Task 0 results, not before ------------------------------------------
# Every None below is a decision that depends on real counts or timings. A later stage refuses to
# run while a value it needs is None, so nothing downstream can start on a guess.
PLAN = {
    "openfake_splits": ("train", "validation"),  # which OpenFake `core` splits are read
    "openfake_shards": None,          # how many shards to read, in a fixed seeded order
    "keep_fraction": None,            # {generator: share of its rows kept}, to thin the biggest generators
    "artifact_per_folder": None,      # images taken per ArtiFact generator folder
    "ffhq_images": None,              # FFHQ thumbnails taken (0 drops the source)
    "partition_fractions": {"train": 0.6, "val": 0.1, "known_test": 0.1, "holdout": 0.2},
    "caps": None,                     # max images per generator per partition
    "phash_tolerance": None,          # dedupe distance, chosen from the real distance histogram
    "min_unseen_fakes": None,         # smallest unseen test a family needs to be held out
    "old_families": None,             # families that count as "old" for the old-only ablation
}
if SMOKE_TEST:
    PLAN.update(openfake_shards=99, keep_fraction={}, artifact_per_folder=10, ffhq_images=12,
                caps={"fake": {"train": 40, "val": 20, "known_test": 20, "holdout": 30},
                      "real": {"train": 200, "val": 60, "known_test": 60, "holdout": 80}},
                phash_tolerance=4, min_unseen_fakes=3, old_families=["sd-1x-2x", "stylegan", "ddpm"])

RESULTS_PATH = OUTPUT_DIR / "results_v1.json"
PREFLIGHT_PATH = OUTPUT_DIR / "preflight_task0.json"
RESULTS = {"smoke_test": SMOKE_TEST, "stage": STAGE, "not_run": {}}
PREFLIGHT = {"smoke_test": SMOKE_TEST, "failed_steps": {}}
SECRETS = []  # filled with the token so it can be scrubbed from anything written or printed


def stage_on(name):
    return STAGES.index(name) <= STAGES.index(STAGE)


def require(*keys):
    missing = [k for k in keys if PLAN.get(k) is None]
    if missing:
        raise RuntimeError(f"PLAN{missing} not decided yet. These are set from the Task 0 results "
                           "(see docs/design_decisions.md, Phase 3) -- run STAGE='task0' first and report it back.")


def save_results(key, value):
    RESULTS[key] = value
    save_json(RESULTS_PATH, RESULTS, SECRETS)


def save_preflight(key, value):
    PREFLIGHT[key] = value
    save_json(PREFLIGHT_PATH, PREFLIGHT, SECRETS)


def elapsed_hours():
    return (time.time() - RUN_START) / 3600


def within_budget(step):
    if elapsed_hours() < TIME_BUDGET_HOURS:
        return True
    RESULTS["not_run"][step] = f"time budget: {elapsed_hours():.1f} h elapsed of {TIME_BUDGET_HOURS} h"
    save_results("not_run", RESULTS["not_run"])
    print(f"STOPPED {step}: {RESULTS['not_run'][step]}")
    return False


def preflight_step(name, fn):
    """Run one pre-flight check. A failure is recorded and printed, and the next check still runs."""
    print(f"\n=== Task 0: {name} ===")
    started = time.time()
    try:
        value = fn()
        save_preflight(name, value)
        print(f"[{name}] done in {time.time() - started:.0f}s")
        return value
    except Exception as exc:  # noqa: BLE001 -- recorded, never hidden
        PREFLIGHT["failed_steps"][name] = scrub_secrets(f"{type(exc).__name__}: {exc}", SECRETS)[:600]
        save_preflight("failed_steps", PREFLIGHT["failed_steps"])
        print(f"[{name}] FAILED: {PREFLIGHT['failed_steps'][name]}")
        print(scrub_secrets(traceback.format_exc(), SECRETS)[-1500:])
        return None


def disk_report(paths=("/kaggle/working", "/kaggle/temp", "/tmp", ".")):
    out = {}
    for p in paths:
        if os.path.isdir(p):
            usage = shutil.disk_usage(p)
            out[p] = {"total_gb": round(usage.total / 1e9, 1), "used_gb": round(usage.used / 1e9, 1),
                      "free_gb": round(usage.free / 1e9, 1)}
    return out


print("device:", DEVICE, "| SMOKE_TEST:", SMOKE_TEST, "| STAGE:", STAGE, "| torch", torch.__version__)

## Shared code (inlined from `engines/`)

The Voice Engine's `evaluate.py` comes first: the Visual Engine reuses its stratified bootstrap and
EER (the code fixed for issue #16) instead of writing them again.

In [ ]:
# ---- inlined from engines/voice/evaluate.py (edit the module, then rebuild) ----
"""EER, DET curve and the silence-leakage probe.

Convention throughout: score = "how bonafide is this", label 1 = bonafide (the target class).
  FAR (false accept rate) = spoof clips scored above threshold  -> accepted as bonafide
  FRR (false reject rate)  = bonafide clips scored below threshold -> rejected as spoof
EER is where those two curves cross, found by linear interpolation between the two bracketing
operating points rather than by taking the nearest point, which is only an approximation.
"""
import numpy as np
from sklearn.metrics import roc_curve


def compute_eer(scores, labels, positive_label=1):
    """Equal Error Rate. Returns eer, the threshold at it, and FAR/FRR there."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    y = (labels == positive_label).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return {"eer": float("nan"), "threshold": float("nan"), "far": float("nan"),
                "frr": float("nan"), "note": "only one class present; EER undefined"}

    fpr, tpr, thresholds = roc_curve(y, scores)
    far, frr = fpr, 1.0 - tpr  # FAR rises with threshold-looseness, FRR falls
    diff = far - frr
    sign_change = np.flatnonzero(np.sign(diff[:-1]) * np.sign(diff[1:]) < 0)
    if len(sign_change) == 0:  # curves touch rather than cross (ties/small samples)
        idx = int(np.nanargmin(np.abs(diff)))
        return {"eer": float((far[idx] + frr[idx]) / 2), "threshold": float(thresholds[idx]),
                "far": float(far[idx]), "frr": float(frr[idx]), "interpolated": False}
    i = int(sign_change[0])
    # Linear interpolation of the crossing between operating points i and i+1.
    weight = diff[i] / (diff[i] - diff[i + 1])
    eer = float(far[i] + weight * (far[i + 1] - far[i]))
    frr_at = float(frr[i] + weight * (frr[i + 1] - frr[i]))
    threshold = float(thresholds[i] + weight * (thresholds[i + 1] - thresholds[i]))
    return {"eer": (eer + frr_at) / 2, "threshold": threshold, "far": eer, "frr": frr_at,
            "interpolated": True}


def det_curve_points(scores, labels, positive_label=1, max_points=512):
    """FAR/FRR pairs for plotting a DET curve, thinned to at most `max_points`."""
    y = (np.asarray(labels) == positive_label).astype(int)
    fpr, tpr, thresholds = roc_curve(y, np.asarray(scores, dtype=np.float64))
    far, frr = fpr, 1.0 - tpr
    if len(far) > max_points:
        keep = np.unique(np.linspace(0, len(far) - 1, max_points).astype(int))
        far, frr, thresholds = far[keep], frr[keep], thresholds[keep]
    return {"far": far.tolist(), "frr": frr.tolist(), "thresholds": thresholds.tolist()}


def per_class_accuracy(scores, labels, threshold, positive_label=1):
    scores, labels = np.asarray(scores), np.asarray(labels)
    predicted_positive = scores > threshold
    is_positive = labels == positive_label
    return {
        "threshold": float(threshold),
        "bonafide_accuracy": float(predicted_positive[is_positive].mean()) if is_positive.any() else None,
        "spoof_accuracy": float((~predicted_positive[~is_positive]).mean()) if (~is_positive).any() else None,
        "overall_accuracy": float((predicted_positive == is_positive).mean()),
        "n_bonafide": int(is_positive.sum()),
        "n_spoof": int((~is_positive).sum()),
    }


def _stratified_resample(labels, rng, positive_label=1):
    """Resample with replacement *within* each class, so every resample keeps the original class
    counts -- a plain resample could drop the (small) bonafide class and leave EER undefined."""
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    return np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])


def bootstrap_eer_ci(scores, labels, n_resamples=1000, seed=0, alpha=0.05, positive_label=1):
    """Percentile bootstrap CI for EER, stratified by class."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = _stratified_resample(labels, rng, positive_label)
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    samples = np.array(samples)
    return {
        "eer": compute_eer(scores, labels, positive_label)["eer"],
        "ci_low": float(np.quantile(samples, alpha / 2)),
        "ci_high": float(np.quantile(samples, 1 - alpha / 2)),
        "ci_width": float(np.quantile(samples, 1 - alpha / 2) - np.quantile(samples, alpha / 2)),
        "bootstrap_std": float(samples.std()),
        "n_resamples": n_resamples,
        "n_bonafide": int((labels == positive_label).sum()),
        "n_spoof": int((labels != positive_label).sum()),
    }


def bootstrap_ci_at_counts(scores, labels, n_bonafide, n_spoof, n_resamples=1000, seed=0, alpha=0.05,
                           positive_label=1):
    """CI width the same scores would have had with a smaller test set of the given class counts.
    Used to say how much the larger Phase 2B test set narrowed the CI versus Phase 2's, on the same
    model, instead of comparing CIs from different models."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = np.concatenate([rng.choice(pos, n_bonafide), rng.choice(neg, n_spoof)])
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    low, high = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
    return {"n_bonafide": int(n_bonafide), "n_spoof": int(n_spoof), "ci_low": float(low),
            "ci_high": float(high), "ci_width": float(high - low)}


def bootstrap_eer_difference(scores_a, labels_a, scores_b, labels_b, n_resamples=1000, seed=0,
                             alpha=0.05, paired=False, positive_label=1):
    """Percentile bootstrap CI for EER(a) - EER(b).

    paired=True: a and b are two models scored on the *same* clips in the same order, so each resample
    draws one set of indices used for both -- this cancels the test-set sampling noise the two share.
    paired=False: two different test sets (e.g. a random split vs. a speaker-disjoint split), each
    resampled independently.
    """
    sa, la = np.asarray(scores_a, dtype=np.float64), np.asarray(labels_a)
    sb, lb = np.asarray(scores_b, dtype=np.float64), np.asarray(labels_b)
    if paired and (len(sa) != len(sb) or not np.array_equal(la, lb)):
        raise ValueError("paired bootstrap needs both models scored on the same clips in the same order")
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        ia = _stratified_resample(la, rng, positive_label)
        ib = ia if paired else _stratified_resample(lb, rng, positive_label)
        diffs.append(compute_eer(sa[ia], la[ia], positive_label)["eer"] -
                     compute_eer(sb[ib], lb[ib], positive_label)["eer"])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = compute_eer(sa, la, positive_label)["eer"] - compute_eer(sb, lb, positive_label)["eer"]
    return {"difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "paired": paired, "n_resamples": n_resamples}


def speaker_leakage_call(diff):
    """Phase 2 printed "speaker leakage" for any positive difference, including +0.0009. The rule is
    now: only a CI that excludes zero counts. `diff` = EER(speaker-disjoint) - EER(random split), so a
    positive, significant difference means the random split flatters the model."""
    ci = f"{diff['difference']:+.4f} EER, 95% CI [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}]"
    if not diff["excludes_zero"]:
        return {"verdict": "not_detectable",
                "text": f"no leakage detectable at this sample size ({ci}; the CI includes zero)"}
    if diff["difference"] > 0:
        return {"verdict": "leakage_detected",
                "text": f"speaker leakage detected: the random split looks better ({ci}, CI excludes zero)"}
    return {"verdict": "random_split_worse",
            "text": f"the random split is significantly *worse* ({ci}) -- not leakage; investigate the splits"}


def per_attack_eer(scores, labels, attacks, positive_label=1):
    """EER of all bonafide clips against each spoofing attack separately."""
    scores, labels, attacks = np.asarray(scores), np.asarray(labels), np.asarray(attacks)
    bonafide = labels == positive_label
    out = {}
    for attack in sorted(set(attacks[~bonafide].tolist())):
        mask = bonafide | (attacks == attack)
        out[attack] = {"eer": compute_eer(scores[mask], labels[mask], positive_label)["eer"],
                       "n_spoof": int((attacks == attack).sum())}
    return out


def evaluate_scores(scores, labels, positive_label=1):
    eer = compute_eer(scores, labels, positive_label)
    return {
        **eer,
        "per_class_at_eer_threshold": per_class_accuracy(scores, labels, eer["threshold"], positive_label),
        "det_curve": det_curve_points(scores, labels, positive_label),
    }


def duration_leakage_probe(stats, labels, positive_label=1):
    """How much can clip timing alone tell you about the label?

    Scores each clip by a duration measure and computes the EER of that single feature. An EER far
    from 50% means the dataset's recording/segmentation artifacts leak the label, independent of any
    spoofing artifact -- the known ASVspoof silence-duration shortcut. Both directions are tried per
    measure, since which sign favours bonafide is itself an empirical question; the reported EER is
    the better (lower) of the two, i.e. the strongest cheat available from that measure.
    """
    labels = np.asarray(labels)
    out = {}
    for key in ("total_s", "lead_silence_s", "trail_silence_s", "speech_s", "silence_ratio"):
        values = np.array([s[key] for s in stats], dtype=np.float64)
        if np.allclose(values, values[0]):
            out[key] = {"eer": float("nan"), "note": "constant measure"}
            continue
        forward = compute_eer(values, labels, positive_label)["eer"]
        backward = compute_eer(-values, labels, positive_label)["eer"]
        better = min(forward, backward)
        y = (labels == positive_label).astype(float)
        out[key] = {
            "eer": float(better),
            "direction": "higher_is_bonafide" if forward <= backward else "lower_is_bonafide",
            "point_biserial_r": float(np.corrcoef(values, y)[0, 1]),
            "mean_bonafide": float(values[labels == positive_label].mean()),
            "mean_spoof": float(values[labels != positive_label].mean()),
        }
    return out


def leakage_verdict(probe, strong=0.35, mild=0.45):
    """Turn the probe into a stated finding. `strong`/`mild` are EER thresholds: the further below
    50% a single duration measure gets, the more label information timing alone carries."""
    usable = {k: v["eer"] for k, v in probe.items() if not np.isnan(v["eer"])}
    if not usable:
        return {"level": "unknown", "worst_measure": None, "worst_eer": None,
                "text": "No usable duration measure; leakage could not be assessed."}
    worst_measure = min(usable, key=usable.get)
    worst = usable[worst_measure]
    level = "strong" if worst < strong else "mild" if worst < mild else "none"
    text = {
        "strong": f"Timing alone separates the classes well ({worst_measure} EER {worst:.4f}), so this data "
                  f"carries the known ASVspoof silence-duration shortcut.",
        "mild": f"Timing alone is weakly predictive ({worst_measure} EER {worst:.4f}) -- some leakage, but far "
                f"from a free solution.",
        "none": f"No meaningful timing leakage: the best duration measure only reaches EER {worst:.4f} "
                f"({worst_measure}), close to the 0.5 of a coin flip.",
    }[level]
    return {"level": level, "worst_measure": worst_measure, "worst_eer": float(worst), "text": text}

In [ ]:
# ---- inlined from engines/visual/preprocess.py (edit the module, then rebuild) ----
"""Image validation, file properties, perceptual hash, the shared pixel pipeline, CLIP preprocessing,
training-time augmentation and the fixed test-time degradations.

**One pixel pipeline for both classes.** Real and fake images arrive with different sizes, formats
and metadata (a JPEG with EXIF from a camera vs. a 1024x1024 PNG from a generator). Any of those
differences is a label shortcut. So every image, whatever its source, goes through the same steps
before the encoder sees it: decode -> upright RGB (alpha flattened, EXIF dropped) -> resize the
shorter side to 224 -> centre crop 224x224 -> one JPEG re-encode at a fixed quality. `standardize`
is that pipeline; CLIP's own normalization follows in `clip_pixels`.

Nothing here writes an image to disk.
"""
import io

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps

CLIP_SIZE = 224
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
MIN_SIDE = 64                 # "tiny" = shorter side below this many pixels
STANDARD_JPEG_QUALITY = 95    # the one re-encode every image gets in `standardize`
DEGRADATIONS = ("jpeg90", "jpeg70", "jpeg50", "resize0.5", "centercrop0.7")

# The IJG reference luminance quantization table (quality 50). Only its sum is used, so its order
# does not matter.
_IJG_LUMA_SUM = 3688


def _flatten_to_rgb(img):
    """Any PIL mode -> RGB. Transparency is composited on white rather than dropped, because the
    colour hidden under transparent pixels is arbitrary."""
    if img.mode in ("I;16", "I;16B", "I;16L", "I", "F"):
        arr = np.asarray(img, dtype=np.float64)
        top = arr.max() if arr.size and arr.max() > 0 else 1.0
        img = Image.fromarray((arr / top * 255.0).astype(np.uint8), mode="L")
    if img.mode == "P" and "transparency" in img.info:
        img = img.convert("RGBA")
    if img.mode in ("RGBA", "LA"):
        rgba = img.convert("RGBA")
        background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
        return Image.alpha_composite(background, rgba).convert("RGB")
    return img.convert("RGB")


def is_grayscale_only(rgb):
    """True when the three channels are identical everywhere (a grayscale picture stored as RGB)."""
    small = np.asarray(rgb.resize((64, 64), Image.BILINEAR), dtype=np.int16)
    return bool((small[..., 0] == small[..., 1]).all() and (small[..., 1] == small[..., 2]).all())


def estimate_jpeg_quality(img):
    """IJG quality (1-100) estimated from a JPEG's luminance quantization table, or None for other
    formats. Uses the table's sum against the reference table's, so it does not depend on whether
    the library returns the table in zigzag or natural order."""
    tables = getattr(img, "quantization", None)
    if img.format != "JPEG" or not tables:
        return None
    luma = np.asarray(list(tables[min(tables)]), dtype=np.float64)
    scale = 100.0 * luma.sum() / _IJG_LUMA_SUM
    quality = (200.0 - scale) / 2.0 if scale <= 100.0 else 5000.0 / scale
    return float(min(100.0, max(1.0, round(quality))))


def decode_image(data, min_side=MIN_SIDE):
    """Bytes -> a validated record. Never raises on a bad file: `ok` is False and `reason` says why
    (`corrupt`, `tiny`, `grayscale`). `rgb` is the upright RGB image, or None when not ok.

    The file properties (`format`, `jpeg_quality`, `has_exif`, `file_size`, `width`, `height`) are
    those of the file *as received* -- they feed the shortcut baseline, not the model.
    """
    record = {"ok": False, "reason": None, "rgb": None, "width": None, "height": None, "format": None,
              "mode": None, "jpeg_quality": None, "file_size": len(data) if data is not None else 0,
              "has_exif": None}
    try:
        img = Image.open(io.BytesIO(data))
        record.update(format=img.format, mode=img.mode, width=img.size[0], height=img.size[1])
        record["jpeg_quality"] = estimate_jpeg_quality(img)
        record["has_exif"] = bool(img.getexif()) or "exif" in img.info
        img.load()
        grayscale_mode = img.mode in ("L", "LA", "1", "I;16", "I;16B", "I;16L", "I", "F")
        rgb = _flatten_to_rgb(ImageOps.exif_transpose(img))
    except Exception as exc:  # noqa: BLE001 -- any decode failure is a corrupt file, recorded not raised
        record["reason"] = "corrupt"
        record["error"] = f"{type(exc).__name__}: {exc}"[:200]
        return record
    record.update(width=rgb.size[0], height=rgb.size[1])
    if min(rgb.size) < min_side:
        record["reason"] = "tiny"
    elif grayscale_mode or is_grayscale_only(rgb):
        record["reason"] = "grayscale"
    else:
        record.update(ok=True, rgb=rgb)
    return record


def validate_image(data, min_side=MIN_SIDE):
    """(ok, reason) only -- the cleaning rule of Task 2 in one call."""
    record = decode_image(data, min_side)
    return record["ok"], record["reason"]


# ---------------------------------------------------------------------------------------------------
# Perceptual hash
# ---------------------------------------------------------------------------------------------------
def phash64(rgb):
    """64-bit DCT perceptual hash (the usual pHash: 32x32 grayscale -> 2-D DCT -> the 8x8 lowest
    frequencies -> one bit per coefficient, above or below their median). Returned as a Python int.
    Resizing or re-encoding an image flips only a few bits, which is what near-duplicate search uses."""
    from scipy.fft import dct

    gray = np.asarray(rgb.convert("L").resize((32, 32), Image.LANCZOS), dtype=np.float64)
    low = dct(dct(gray, axis=0, norm="ortho"), axis=1, norm="ortho")[:8, :8]
    bits = (low > np.median(low)).flatten()
    return int(np.packbits(bits).view(">u8")[0])


# ---------------------------------------------------------------------------------------------------
# The shared pipeline and CLIP preprocessing
# ---------------------------------------------------------------------------------------------------
def jpeg_bytes(rgb, quality):
    buffer = io.BytesIO()
    rgb.save(buffer, format="JPEG", quality=int(quality))
    return buffer.getvalue()


def jpeg_roundtrip(rgb, quality):
    return Image.open(io.BytesIO(jpeg_bytes(rgb, quality))).convert("RGB")


def resize_center_crop(rgb, size=CLIP_SIZE):
    """CLIP's own geometry: shorter side to `size` (bicubic), then a centre crop of size x size.
    The longer side is rounded down, as CLIP's own processor does, so the two agree pixel for pixel."""
    w, h = rgb.size
    new_w, new_h = (size, max(size, int(size * h / w))) if w <= h else (max(size, int(size * w / h)), size)
    resized = rgb.resize((new_w, new_h), Image.BICUBIC)
    left, top = (new_w - size) // 2, (new_h - size) // 2
    return resized.crop((left, top, left + size, top + size))


def standardize(rgb, jpeg_quality=STANDARD_JPEG_QUALITY):
    """The shared pipeline: 224x224 RGB after one JPEG re-encode. Returns (image, jpeg byte count).
    The byte count is the only file property left that still varies between images, so it is what
    the post-pipeline shortcut baseline is run on."""
    fixed = resize_center_crop(rgb)
    if jpeg_quality is None:
        return fixed, None
    data = jpeg_bytes(fixed, jpeg_quality)
    return Image.open(io.BytesIO(data)).convert("RGB"), len(data)


def clip_pixels(rgb):
    """A 224x224 RGB image -> float32 tensor [3, 224, 224], scaled to [0, 1] and normalized with
    CLIP's mean and std. Images of another size get CLIP's resize + centre crop first."""
    if rgb.size != (CLIP_SIZE, CLIP_SIZE):
        rgb = resize_center_crop(rgb)
    arr = np.asarray(rgb, dtype=np.float32) / 255.0
    arr = (arr - np.asarray(CLIP_MEAN, dtype=np.float32)) / np.asarray(CLIP_STD, dtype=np.float32)
    return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1)))


def preprocess_image(rgb, jpeg_quality=STANDARD_JPEG_QUALITY):
    """Upright RGB of any size -> the model input. Deterministic: same input, same tensor."""
    return clip_pixels(standardize(rgb, jpeg_quality)[0])


def inspect_image(data, min_side=MIN_SIDE, jpeg_quality=STANDARD_JPEG_QUALITY):
    """Everything the metadata table needs from one image, without keeping its pixels: validity,
    the file properties as received, the perceptual hash, and the byte count after the shared
    pipeline (`pipeline_file_size`)."""
    record = decode_image(data, min_side)
    rgb = record.pop("rgb")
    record.update(phash=None, pipeline_file_size=None)
    if record["ok"]:
        record["phash"] = phash64(rgb)
        record["pipeline_file_size"] = standardize(rgb, jpeg_quality)[1]
    return record


# ---------------------------------------------------------------------------------------------------
# Training-time augmentation (train split only) and fixed test-time degradations
# ---------------------------------------------------------------------------------------------------
_INTERPOLATIONS = (Image.BILINEAR, Image.BICUBIC, Image.LANCZOS)


def view_rng(image_id, view, seed=0):
    """A generator that depends only on (image, view, seed), so re-running an interrupted
    extraction reproduces exactly the same augmented views."""
    import hashlib

    digest = hashlib.sha256(f"{seed}|{view}|{image_id}".encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], "big"))


def augment_view(rgb, rng, scale=(0.5, 1.5), crop_area=(0.7, 1.0), jpeg_quality=(30, 100), colour=0.2):
    """One augmented view for the CLIP branch: random resize (random interpolation), random crop
    of 70-100% of the area, JPEG at a random quality, mild brightness/contrast. No rotation and no
    flip (rotation resamples pixels; a flip is wrong for images containing text). Ranges are the
    handbook's Section 21 starting points."""
    w, h = rgb.size
    factor = rng.uniform(*scale)
    new_w, new_h = max(MIN_SIDE, round(w * factor)), max(MIN_SIDE, round(h * factor))
    out = rgb.resize((new_w, new_h), _INTERPOLATIONS[int(rng.integers(len(_INTERPOLATIONS)))])
    side = float(np.sqrt(rng.uniform(*crop_area)))
    crop_w, crop_h = max(MIN_SIDE // 2, round(new_w * side)), max(MIN_SIDE // 2, round(new_h * side))
    left = int(rng.integers(0, new_w - crop_w + 1))
    top = int(rng.integers(0, new_h - crop_h + 1))
    out = out.crop((left, top, left + crop_w, top + crop_h))
    out = ImageEnhance.Brightness(out).enhance(1.0 + rng.uniform(-colour, colour))
    out = ImageEnhance.Contrast(out).enhance(1.0 + rng.uniform(-colour, colour))
    return jpeg_roundtrip(out, int(rng.integers(jpeg_quality[0], jpeg_quality[1] + 1)))


def degrade(rgb, name):
    """The fixed, documented degradations of the robustness suites, applied to the original image
    before the shared pipeline: JPEG at quality 90/70/50, a 0.5x resize, a centre crop keeping 70%
    of each side."""
    if name.startswith("jpeg"):
        return jpeg_roundtrip(rgb, int(name[4:]))
    w, h = rgb.size
    if name.startswith("resize"):
        factor = float(name[6:])
        return rgb.resize((max(1, round(w * factor)), max(1, round(h * factor))), Image.BICUBIC)
    if name.startswith("centercrop"):
        keep = float(name[10:])
        crop_w, crop_h = max(1, round(w * keep)), max(1, round(h * keep))
        left, top = (w - crop_w) // 2, (h - crop_h) // 2
        return rgb.crop((left, top, left + crop_w, top + crop_h))
    raise ValueError(f"unknown degradation: {name}")


def variant_image(rgb, variant, image_id, seed=0):
    """`clean`, `aug<k>` or a degradation name -> the image that goes into `standardize`."""
    if variant == "clean":
        return rgb
    if variant.startswith("aug"):
        return augment_view(rgb, view_rng(image_id, variant, seed))
    return degrade(rgb, variant)

In [ ]:
# ---- inlined from engines/visual/data.py (edit the module, then rebuild) ----
"""Data access for the Visual Engine: secrets, sparse Parquet reads over HTTP ranges, folder
datasets, the streaming-feasibility measurements of Task 0, and synthetic data for local tests.

**Why a sparse Parquet reader.** OpenFake is 3.4 TB of Parquet shards of about 5 GB each, and each
row group holds about 70 images in about 128 MB (as read from two shard footers on 2026-10-07; Task 0
re-measures this on Kaggle). The label/model/prompt columns of a row group are a few kilobytes that
sit right after its image column. `SparseParquet` fetches the footer, works out the byte range of
exactly the column chunks asked for, and downloads only those -- so the whole dataset's metadata can
be counted without touching an image, and one row group's images can be fetched without the rest of
its shard. Nothing is ever written to disk.
"""
import csv
import hashlib
import io
import json
import os
import re
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image


OPENFAKE_REPO = "ComplexDataLab/OpenFake"
DEEPFAKE_EVAL_REPO = "nuriachandra/Deepfake-Eval-2024"
HF_SECRET_NAME = "HF_TOKEN"
REAL, FAKE = 0, 1
META_COLUMNS = ("prompt", "label", "model", "type", "release_date", "row_id")
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".webp", ".bmp")


# ---------------------------------------------------------------------------------------------------
# Secrets: read from Kaggle Secrets, never printed, never saved
# ---------------------------------------------------------------------------------------------------
def read_hf_token():
    """The Hugging Face token from the Kaggle secret `HF_TOKEN` (or the environment variable of the
    same name), or None. Callers must not print it or store it in results."""
    token = None
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:  # noqa: BLE001 -- not on Kaggle, or the secret is not attached to this notebook
        token = None
    return token or os.environ.get(HF_SECRET_NAME) or None


def scrub_secrets(value, secrets):
    """A copy of `value` (nested dicts/lists/strings) with every secret replaced by `<redacted>`.
    Everything written to a results file or printed from an exception goes through this."""
    secrets = [s for s in (secrets or []) if s]
    if isinstance(value, str):
        for secret in secrets:
            value = value.replace(secret, "<redacted>")
        return value
    if isinstance(value, dict):
        return {scrub_secrets(k, secrets): scrub_secrets(v, secrets) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [scrub_secrets(v, secrets) for v in value]
    return value


# ---------------------------------------------------------------------------------------------------
# Byte-range readers
# ---------------------------------------------------------------------------------------------------
def _with_retries(fn, attempts=4, wait=2.0):
    for attempt in range(attempts):
        try:
            return fn()
        except Exception:  # noqa: BLE001 -- network reads are retried, then re-raised
            if attempt == attempts - 1:
                raise
            time.sleep(wait * (attempt + 1))


class FsRangeReader:
    """Byte ranges through any fsspec filesystem (local files in tests, `HfFileSystem` on Kaggle)."""

    def __init__(self, fs):
        self.fs = fs
        self.requests = 0
        self.bytes_read = 0

    def read(self, path, start, end):
        data = _with_retries(lambda: self.fs.cat_file(path, start=start, end=end))
        self.requests += 1
        self.bytes_read += len(data)
        return data


class HttpRangeReader:
    """Byte ranges with plain HTTP `Range` requests against a Hugging Face dataset repository.
    Takes the same `datasets/<repo>/<file>` paths as `HfFileSystem`.

    The Hub answers a file request with a redirect to a signed storage URL. That URL is remembered
    per file, so reading the ~60 row groups of a shard costs the Hub one request, not sixty (the Hub
    limits how many requests an account may make; the storage host is not part of that limit and
    never sees the token). The token is sent to the Hub only, in the request header.
    """

    REDIRECTS = (301, 302, 303, 307, 308)

    def __init__(self, repo=OPENFAKE_REPO, token=None, revision="main", session=None, timeout=600):
        self.repo, self.revision, self.timeout = repo, revision, timeout
        self._headers = {"Authorization": f"Bearer {token}"} if token else {}
        if session is None:
            from huggingface_hub.utils import get_session

            session = get_session()
        self._session = session
        self._storage_url = {}
        self.requests = 0
        self.hub_requests = 0
        self.bytes_read = 0

    def _hub_url(self, path):
        from urllib.parse import quote

        prefix = f"datasets/{self.repo}/"
        filename = path[len(prefix):] if path.startswith(prefix) else path
        return f"https://huggingface.co/datasets/{self.repo}/resolve/{quote(self.revision, safe='')}/{quote(filename)}"

    def _get(self, url, headers):
        try:
            return self._session.get(url, headers=headers, follow_redirects=False, timeout=self.timeout)
        except TypeError:  # an older huggingface_hub whose session is `requests`
            return self._session.get(url, headers=headers, allow_redirects=False, timeout=self.timeout)

    def _resolve(self, path):
        """Follow the Hub's redirects (asking for one byte) to the storage URL of a file."""
        from urllib.parse import urljoin, urlparse

        url = self._hub_url(path)
        for _ in range(6):
            if not urlparse(url).netloc.endswith("huggingface.co"):
                return url
            response = self._get(url, {**self._headers, "Range": "bytes=0-0"})
            self.hub_requests += 1
            if response.status_code == 429:
                time.sleep(float(response.headers.get("Retry-After", 30)))
                continue
            if response.status_code not in self.REDIRECTS:
                if response.status_code in (200, 206):
                    return url   # the Hub serves this file itself
                raise OSError(f"HTTP {response.status_code} resolving {path}")
            url = urljoin(url, response.headers["Location"])
        raise OSError(f"too many redirects resolving {path}")

    def read(self, path, start, end):
        from urllib.parse import urlparse

        def fetch():
            url = self._storage_url.get(path)
            if url is None:
                url = self._storage_url[path] = self._resolve(path)
            headers = {"Range": f"bytes={start}-{end - 1}"}
            if urlparse(url).netloc.endswith("huggingface.co"):
                headers.update(self._headers)
            response = self._get(url, headers)
            if response.status_code == 429:
                time.sleep(float(response.headers.get("Retry-After", 30)))
                raise OSError(f"HTTP 429 for {path}")
            if response.status_code not in (200, 206):
                self._storage_url.pop(path, None)   # a signed URL that expired: resolve again on the retry
                raise OSError(f"HTTP {response.status_code} for a range read of {path}")
            data = response.content
            if len(data) != end - start:
                raise OSError(f"range read of {path} returned {len(data)} bytes, expected {end - start}")
            return data

        data = _with_retries(fetch)
        self.requests += 1
        self.bytes_read += len(data)
        return data


class _SparseFile(io.IOBase):
    """A read-only file made of the byte ranges fetched so far. A read outside them is fetched on
    demand and counted in `misses`, so tests can assert the range planning was exact."""

    def __init__(self, reader, path, size):
        self._reader, self._path, self._size = reader, path, int(size)
        self._parts = []  # (start, bytes)
        self._pos = 0
        self.misses = 0

    def add(self, start, data):
        self._parts.append((int(start), data))

    def clear(self, keep_from=None):
        self._parts = [p for p in self._parts if keep_from is not None and p[0] >= keep_from]

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self._pos

    def seek(self, offset, whence=0):
        self._pos = offset if whence == 0 else self._pos + offset if whence == 1 else self._size + offset
        return self._pos

    def read(self, n=-1):
        start = self._pos
        end = self._size if n is None or n < 0 else min(self._size, start + n)
        if end <= start:
            return b""
        for part_start, data in self._parts:
            if part_start <= start and end <= part_start + len(data):
                self._pos = end
                return bytes(data[start - part_start:end - part_start])
        self.misses += 1
        data = self._reader.read(self._path, start, end)
        self._pos = end
        return data


class SparseParquet:
    """One Parquet file read by byte range. `read(columns, row_groups)` downloads only the column
    chunks it needs."""

    def __init__(self, reader, path, size, footer_guess=1 << 20):
        import pyarrow.parquet as pq

        self.reader, self.path, self.size = reader, path, int(size)
        tail_start = max(0, self.size - footer_guess)
        tail = reader.read(path, tail_start, self.size)
        footer_len = int.from_bytes(tail[-8:-4], "little")
        # The Parquet reader starts by reading the last 64 KB, so the cached tail must cover that and the footer.
        needed = max(footer_len + 8, 1 << 16)
        if needed > len(tail) and tail_start > 0:
            tail_start = max(0, self.size - needed)
            tail = reader.read(path, tail_start, self.size)
        self._tail_start = tail_start
        self.file = _SparseFile(reader, path, self.size)
        self.file.add(tail_start, tail)
        self.parquet = pq.ParquetFile(self.file)
        self.meta = self.parquet.metadata
        self.columns = list(self.parquet.schema_arrow.names)

    @property
    def num_row_groups(self):
        return self.meta.num_row_groups

    def row_group_rows(self):
        return [self.meta.row_group(i).num_rows for i in range(self.meta.num_row_groups)]

    def _chunks(self, row_group, columns):
        group = self.meta.row_group(row_group)
        for i in range(group.num_columns):
            chunk = group.column(i)
            name = chunk.path_in_schema
            if any(name == c or name.startswith(c + ".") for c in columns):
                offset = chunk.data_page_offset
                if chunk.has_dictionary_page and chunk.dictionary_page_offset is not None:
                    offset = min(offset, chunk.dictionary_page_offset)
                yield offset, offset + chunk.total_compressed_size

    def column_bytes(self, columns, row_groups=None):
        """Compressed bytes of `columns` per row group (from the footer alone, nothing downloaded)."""
        groups = range(self.num_row_groups) if row_groups is None else row_groups
        return [sum(end - start for start, end in self._chunks(g, columns)) for g in groups]

    def read(self, columns, row_groups=None, max_gap=1 << 16):
        """An Arrow table of `columns` for `row_groups` (all by default)."""
        groups = list(range(self.num_row_groups)) if row_groups is None else list(row_groups)
        ranges = sorted(r for g in groups for r in self._chunks(g, columns))
        merged = []
        for start, end in ranges:
            if merged and start - merged[-1][1] <= max_gap:
                merged[-1][1] = max(merged[-1][1], end)
            else:
                merged.append([start, end])
        self.file.clear(keep_from=self._tail_start)
        for start, end in merged:
            self.file.add(start, self.reader.read(self.path, start, end))
        table = self.parquet.read_row_groups(groups, columns=list(columns))
        self.file.clear(keep_from=self._tail_start)
        return table


def image_bytes_column(table, column="image"):
    """The raw image bytes of an Arrow table's `image` column (a struct with a `bytes` field in
    Hugging Face image datasets, or a plain binary column)."""
    import pyarrow as pa

    array = table.column(column).combine_chunks()
    if pa.types.is_struct(array.type):
        array = array.field("bytes")
    return array.to_pylist()


def list_parquet_shards(fs, root):
    """Every Parquet file under `root` with its size, config (parent folder) and split (file-name
    prefix), sorted by path."""
    shards = []
    for path, info in fs.find(root, detail=True).items():
        if path.endswith(".parquet"):
            name = path.rsplit("/", 1)[-1]
            split = re.match(r"([A-Za-z_]+)", name).group(1) if re.match(r"([A-Za-z_]+)", name) else ""
            shards.append({"path": path, "size": int(info["size"]), "config": path.rsplit("/", 2)[-2],
                           "split": split})
    return sorted(shards, key=lambda s: s["path"])


# ---------------------------------------------------------------------------------------------------
# Identifiers
# ---------------------------------------------------------------------------------------------------
def prompt_id_of(prompt):
    """A stable id for a prompt (lower-cased, whitespace collapsed), or None for an empty one.
    Images made from the same prompt -- and the real photo whose caption the prompt came from --
    share it, so it is one of the keys splits are grouped by."""
    text = " ".join(str(prompt or "").lower().split())
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16] if text else None


def stable_fraction(key, seed=0):
    """A deterministic number in [0, 1) from a string key: the same key always lands in the same
    place, on any machine, in any session."""
    digest = hashlib.sha256(f"{seed}|{key}".encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") / 2.0 ** 64


# ---------------------------------------------------------------------------------------------------
# Task 0: metadata-only scan and the timed streamed sample
# ---------------------------------------------------------------------------------------------------
def scan_shard_metadata(reader, shard, columns=META_COLUMNS):
    """Counts for one shard from its small columns only -- no image is downloaded. Returns
    aggregate counts plus per-row prompt ids (used in memory for group statistics, never saved)."""
    started = time.time()
    sparse = SparseParquet(reader, shard["path"], shard["size"])
    wanted = [c for c in columns if c in sparse.columns]
    table = sparse.read(wanted).to_pydict()
    n = sparse.meta.num_rows
    column = lambda name: table.get(name, [None] * n)  # noqa: E731
    labels, models = column("label"), column("model")
    return {
        "path": shard["path"], "config": shard["config"], "split": shard["split"], "size": shard["size"],
        "rows": n, "row_groups": sparse.num_row_groups, "row_group_rows": sparse.row_group_rows(),
        "image_bytes": int(sum(sparse.column_bytes(["image"]))),
        "columns": sparse.columns,
        "counts": Counter(zip(labels, models, column("type"), column("release_date"))),
        "label": labels, "model": models,
        "prompt_id": [prompt_id_of(p) for p in column("prompt")],
        "seconds": time.time() - started, "sparse_misses": sparse.file.misses,
    }


def scan_all_shards(reader, shards, threads=16, max_seconds=None, progress_every=25, log=print):
    """`scan_shard_metadata` over many shards in parallel, with a progress line every few shards
    and an optional time budget (the shards not reached are reported, not hidden)."""
    started, done, failed, skipped = time.time(), [], [], []
    with ThreadPoolExecutor(max_workers=threads) as pool:
        futures = [(shard, pool.submit(scan_shard_metadata, reader, shard)) for shard in shards]
        for i, (shard, future) in enumerate(futures):
            if max_seconds is not None and time.time() - started > max_seconds and not future.done():
                future.cancel()
            try:
                if future.cancelled():
                    skipped.append(shard["path"])
                else:
                    done.append(future.result())
            except Exception as exc:  # noqa: BLE001 -- one bad shard is recorded, the scan goes on
                failed.append({"path": shard["path"], "error": f"{type(exc).__name__}: {exc}"[:300]})
            if progress_every and (i + 1) % progress_every == 0:
                log(f"  metadata scan {i + 1}/{len(shards)} shards, {sum(d['rows'] for d in done)} rows, "
                    f"{time.time() - started:.0f}s")
    return {"shards": done, "failed": failed, "skipped_for_time": skipped, "seconds": time.time() - started}


def prompt_group_stats(scan):
    """How prompts tie rows together, from a metadata scan: how many prompts have both a real and a
    fake row, how many span several generators, and how big the groups are."""
    by_prompt = {}
    empty = 0
    for shard in scan["shards"]:
        for label, model, prompt in zip(shard["label"], shard["model"], shard["prompt_id"]):
            if prompt is None:
                empty += 1
                continue
            entry = by_prompt.setdefault(prompt, [0, 0, set()])
            entry[1 if label == "fake" else 0] += 1
            if label == "fake":
                entry[2].add(model)
    sizes = np.array([e[0] + e[1] for e in by_prompt.values()] or [0])
    return {
        "rows_with_empty_prompt": empty,
        "distinct_prompts": len(by_prompt),
        "prompts_with_real_and_fake": sum(1 for e in by_prompt.values() if e[0] and e[1]),
        "prompts_with_fakes_from_2plus_models": sum(1 for e in by_prompt.values() if len(e[2]) >= 2),
        "prompts_with_one_row": int((sizes == 1).sum()),
        "group_size_quantiles": {q: float(np.quantile(sizes, float(q))) for q in ("0.5", "0.9", "0.99", "1.0")},
    }


def time_streamed_sample(reader, shards, max_images=3000, max_seconds=600, decode_threads=4, log=print):
    """Download and decode real row groups from at least two shards and time it. The second shard
    is entered in the middle (a seek), to show a row group can be fetched without the ones before
    it. Images are decoded, hashed and pushed through the shared pipeline, then dropped."""
    started = time.time()
    out = {"images": 0, "image_bytes": 0, "download_seconds": 0.0, "decode_seconds": 0.0, "row_groups": [],
           "invalid": Counter(), "format": Counter(), "mode": Counter(), "sizes": Counter(), "labels": Counter()}
    widths, heights, file_sizes = [], [], []
    with ThreadPoolExecutor(max_workers=decode_threads) as pool:
        for k, shard in enumerate(shards):
            sparse = SparseParquet(reader, shard["path"], shard["size"])
            first = 0 if k == 0 else sparse.num_row_groups // 2
            for group in range(first, sparse.num_row_groups):
                if out["images"] >= max_images * (k + 1) / len(shards) or time.time() - started > max_seconds:
                    break
                t0 = time.time()
                table = sparse.read(["image", "label"], [group])
                blobs = image_bytes_column(table)
                t1 = time.time()
                records = list(pool.map(inspect_image, blobs))
                t2 = time.time()
                nbytes = sum(len(b) for b in blobs)
                out["images"] += len(blobs)
                out["image_bytes"] += nbytes
                out["download_seconds"] += t1 - t0
                out["decode_seconds"] += t2 - t1
                out["row_groups"].append({"shard": k, "row_group": group, "rows": len(blobs), "mb": round(nbytes / 1e6, 1),
                                          "download_s": round(t1 - t0, 2), "decode_s": round(t2 - t1, 2),
                                          "entered_by_seek": bool(k > 0 and group == first)})
                out["labels"].update(table.column("label").to_pylist())
                for r in records:
                    out["format"][r["format"]] += 1
                    out["mode"][r["mode"]] += 1
                    if r["ok"]:
                        widths.append(r["width"])
                        heights.append(r["height"])
                        file_sizes.append(r["file_size"])
                        out["sizes"][f"{r['width']}x{r['height']}"] += 1
                    else:
                        out["invalid"][r["reason"]] += 1
                log(f"  sample: shard {k} row group {group}: {len(blobs)} images, {nbytes / 1e6:.0f} MB, "
                    f"download {t1 - t0:.1f}s, decode {t2 - t1:.1f}s")
            out["sparse_misses"] = out.get("sparse_misses", 0) + sparse.file.misses
    seconds = out["download_seconds"] + out["decode_seconds"]
    out.update(
        wall_seconds=time.time() - started,
        mb_per_second=out["image_bytes"] / 1e6 / out["download_seconds"] if out["download_seconds"] else None,
        decode_images_per_second=out["images"] / out["decode_seconds"] if out["decode_seconds"] else None,
        images_per_second_sequential=out["images"] / seconds if seconds else None,
        mean_mb_per_image=out["image_bytes"] / 1e6 / out["images"] if out["images"] else None,
        width_quantiles=_quantiles(widths), height_quantiles=_quantiles(heights),
        file_size_quantiles=_quantiles(file_sizes), decode_threads=decode_threads,
        sizes=dict(out["sizes"].most_common(12)),
    )
    for key in ("invalid", "format", "mode", "labels"):
        out[key] = {str(k): v for k, v in out[key].items()}
    return out


def _quantiles(values, qs=(0.05, 0.5, 0.95)):
    return {str(q): float(np.quantile(values, q)) for q in qs} if len(values) else {}


def shards_to_fill_quota(per_shard_counts, quota):
    """Taking whole shards in the given order, how many are needed before each generator has
    `quota` rows. `per_shard_counts` is a list (in read order) of {generator: rows}. Returns
    {generator: shards needed, or None if the listed shards never reach the quota}."""
    running, needed = Counter(), {}
    generators = set().union(*[set(c) for c in per_shard_counts]) if per_shard_counts else set()
    for i, counts in enumerate(per_shard_counts, 1):
        running.update(counts)
        for g in generators:
            if g not in needed and running[g] >= quota:
                needed[g] = i
    return {g: needed.get(g) for g in sorted(generators)}


# ---------------------------------------------------------------------------------------------------
# Folder datasets (ArtiFact and the FFHQ thumbnails are attached to the Kaggle notebook as files)
# ---------------------------------------------------------------------------------------------------
def find_kaggle_inputs(base="/kaggle/input", depth=3, max_entries=500):
    """Attached dataset folders, nearest first, down to `depth` levels (Kaggle has mounted datasets
    at different depths over time). A folder with more than `max_entries` entries is listed but not
    opened further, so a folder of a hundred thousand images costs nothing."""
    found, level = [], [str(base)] if os.path.isdir(base) else []
    for _ in range(depth):
        below = []
        for folder in level:
            try:
                with os.scandir(folder) as entries:
                    children = []
                    for n, entry in enumerate(entries):
                        if n >= max_entries:
                            children = []
                            break
                        if entry.is_dir(follow_symlinks=False):
                            children.append(entry.path)
            except OSError:
                continue
            below.extend(sorted(children))
        found.extend(below)
        level = below
    return [f.replace("\\", "/") for f in found]


def match_input(folders, keywords, not_under=None):
    """The first attached folder whose name contains every one of `keywords` (case-insensitive),
    skipping `not_under` and anything inside it."""
    for folder in folders:
        name = Path(folder).name.lower()
        inside = bool(not_under) and (Path(folder).as_posix() + "/").startswith(Path(not_under).as_posix().rstrip("/") + "/")
        if all(k in name for k in keywords) and not inside:
            return folder
    return None


def summarize_folder(root, max_seconds=900, progress_every=200000, log=print):
    """File counts and bytes under `root`, by top-level folder and extension, plus the CSV files
    found. Stops at the time budget and says so."""
    started, root = time.time(), str(root)
    summary = {"root": root, "files": 0, "bytes": 0, "by_top_dir": {}, "csv_files": [], "complete": True}
    stack = [root]
    while stack:
        if time.time() - started > max_seconds:
            summary["complete"] = False
            break
        folder = stack.pop()
        try:
            entries = list(os.scandir(folder))
        except OSError:
            continue
        for entry in entries:
            if entry.is_dir(follow_symlinks=False):
                stack.append(entry.path)
                continue
            rel = os.path.relpath(entry.path, root).replace("\\", "/")
            top = rel.split("/", 1)[0] if "/" in rel else "."
            ext = os.path.splitext(entry.name)[1].lower()
            bucket = summary["by_top_dir"].setdefault(top, {"files": 0, "ext": {}})
            bucket["files"] += 1
            bucket["ext"][ext] = bucket["ext"].get(ext, 0) + 1
            summary["files"] += 1
            summary["bytes"] += entry.stat(follow_symlinks=False).st_size
            if ext == ".csv":
                summary["csv_files"].append(rel)
            if progress_every and summary["files"] % progress_every == 0:
                log(f"  walked {summary['files']} files under {root}, {time.time() - started:.0f}s")
    summary["seconds"] = time.time() - started
    return summary


def describe_csv(path, max_rows=None, max_distinct=40):
    """Column names, row count and the value counts of every low-cardinality column of a CSV."""
    counts, rows, columns = {}, 0, []
    with open(path, newline="", encoding="utf-8", errors="replace") as handle:
        reader = csv.DictReader(handle)
        columns = list(reader.fieldnames or [])
        for row in reader:
            rows += 1
            for name in columns:
                seen = counts.setdefault(name, Counter())
                if seen is not None and len(seen) <= max_distinct:
                    seen[row.get(name)] += 1
            if max_rows and rows >= max_rows:
                break
    return {"columns": columns, "rows": rows,
            "value_counts": {k: dict(v) for k, v in counts.items() if len(v) <= max_distinct}}


def list_images(root):
    """Every image file under `root`, as sorted paths."""
    found = []
    for folder, _, names in os.walk(root):
        found.extend(os.path.join(folder, n) for n in names if n.lower().endswith(IMAGE_EXTENSIONS))
    return sorted(found)


def image_header(path):
    """(width, height, mode, format) from the file header alone -- no pixel decode."""
    try:
        with Image.open(path) as img:
            return img.size[0], img.size[1], img.mode, img.format
    except Exception:  # noqa: BLE001
        return None


def header_summary(paths, threads=8):
    """Size, mode and format counts over image files (header reads only)."""
    with ThreadPoolExecutor(max_workers=threads) as pool:
        headers = list(pool.map(image_header, paths))
    good = [h for h in headers if h is not None]
    return {"files": len(paths), "unreadable": len(headers) - len(good),
            "sizes": dict(Counter(f"{h[0]}x{h[1]}" for h in good).most_common(12)),
            "modes": dict(Counter(h[2] for h in good)), "formats": dict(Counter(h[3] for h in good))}


def index_artifact(root):
    """One row per ArtiFact image from the `metadata.csv` in each generator folder. The layout
    (columns `image_path` and `target`, 0 = real) is the one the dataset's README describes; Task 0
    prints the real one, and this function raises if it differs rather than guessing."""
    rows = []
    for folder in sorted(p for p in Path(root).iterdir() if p.is_dir()):
        meta = folder / "metadata.csv"
        if not meta.exists():
            continue
        with open(meta, newline="", encoding="utf-8", errors="replace") as handle:
            reader = csv.DictReader(handle)
            if not {"image_path", "target"} <= set(reader.fieldnames or []):
                raise ValueError(f"{meta} has columns {reader.fieldnames}, expected image_path and target")
            for row in reader:
                rel = row["image_path"].replace("\\", "/")
                rows.append({"image_id": f"artifact:{folder.name}/{rel}", "source_dataset": "artifact",
                             "generator": folder.name, "is_fake": int(str(row["target"]).strip() != "0"),
                             "category": row.get("category"), "path": str(folder / rel)})
    return rows


def index_ffhq(root):
    """One row per FFHQ thumbnail. Each FFHQ image is a different person's photo, so the file stem
    is both the source image id and the identity id."""
    return [{"image_id": f"ffhq:{Path(p).stem}", "source_dataset": "ffhq", "generator": "ffhq", "is_fake": REAL,
             "source_image_id": f"ffhq:{Path(p).stem}", "identity_id": f"ffhq:{Path(p).stem}", "path": p}
            for p in list_images(root)]


def read_file_bytes(path):
    with open(path, "rb") as handle:
        return handle.read()


# ---------------------------------------------------------------------------------------------------
# Synthetic data (unit tests and the notebook smoke run -- never a reported number)
# ---------------------------------------------------------------------------------------------------
SYNTHETIC_GENERATORS = {  # name -> (type, release date)
    "sd-1.5": ("base", "2022-10"), "sd-2.1": ("base", "2022-12"), "sdxl": ("base", "2023-07"),
    "sdxl-juggernaut": ("finetune", "2023-11"), "flux.1-dev": ("base", "2024-08"),
    "flux.1-schnell": ("base", "2024-08"), "midjourney-6": ("base", "2023-12"),
    "midjourney-7": ("base", "2025-04"), "gpt-image-1": ("base", "2025-04"),
}


def synthetic_picture(rng, fake, size=(160, 128), strength=40.0):
    """A smooth random picture. Fakes get a faint periodic grid on top -- a planted, learnable
    difference, so the heads have something to find."""
    w, h = size
    coarse = rng.uniform(0, 255, size=(6, 6, 3))
    img = Image.fromarray(coarse.astype(np.uint8)).resize((w, h), Image.BICUBIC)
    arr = np.asarray(img, dtype=np.float64) + rng.normal(0, 4.0, size=(h, w, 3))
    if fake:
        yy, xx = np.mgrid[0:h, 0:w]
        arr += strength * (((xx // 4) + (yy // 4)) % 2)[..., None]
    return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))


def _encode(img, fmt, **kwargs):
    buffer = io.BytesIO()
    img.save(buffer, format=fmt, **kwargs)
    return buffer.getvalue()


def make_synthetic_openfake(n_prompts=60, fakes_per_prompt=4, seed=0, with_problems=True):
    """OpenFake-shaped rows: one real photo per prompt and fakes of the same prompt from several
    generators. Reals are JPEGs of varying size with EXIF, fakes are fixed-size PNGs -- the planted
    file-property shortcut the baseline must catch. With `with_problems`, a few rows are an exact
    duplicate, a re-encoded near duplicate, a corrupt file, a tiny image, a grayscale image and an
    RGBA image."""
    rng = np.random.default_rng(seed)
    names = list(SYNTHETIC_GENERATORS)
    rows = []

    def add(image_bytes, label, model, kind, date, prompt):
        rows.append({"image": {"bytes": image_bytes, "path": None}, "prompt": prompt, "label": label,
                     "model": model, "type": kind, "release_date": date, "row_id": f"row{len(rows):06d}",
                     "image_sha256": hashlib.sha256(image_bytes).hexdigest()})

    for p in range(n_prompts):
        prompt = f"A photograph of scene number {p} with people in a public place"
        size = (int(rng.integers(140, 260)), int(rng.integers(120, 220)))
        exif = Image.Exif()
        exif[0x010F] = "SyntheticCam"
        real = _encode(synthetic_picture(rng, False, size), "JPEG", quality=int(rng.integers(70, 96)), exif=exif)
        add(real, "real", "laion" if p % 2 else "pexels", "real", "2021-08", prompt)
        for g in rng.choice(len(names), size=min(fakes_per_prompt, len(names)), replace=False):
            kind, date = SYNTHETIC_GENERATORS[names[g]]
            add(_encode(synthetic_picture(rng, True, (192, 192)), "PNG"), "fake", names[g], kind, date, prompt)
    if with_problems:
        first_real = rows[0]["image"]["bytes"]
        add(first_real, "real", "laion", "real", "2021-08", "an exact duplicate of the first real photo")
        again = _encode(Image.open(io.BytesIO(first_real)).resize((120, 100)), "JPEG", quality=60)
        add(again, "real", "laion", "real", "2021-08", "a re-encoded copy of the first real photo")
        add(b"not an image at all", "fake", names[0], "base", "2022-10", "a corrupt file")
        add(_encode(synthetic_picture(rng, True, (32, 32)), "PNG"), "fake", names[0], "base", "2022-10", "a tiny image")
        add(_encode(synthetic_picture(rng, False, (160, 128)).convert("L"), "PNG"), "real", "laion", "real",
            "2021-08", "a grayscale image")
        add(_encode(synthetic_picture(rng, True, (192, 192)).convert("RGBA"), "PNG"), "fake", names[2], "base",
            "2023-07", "an image with an alpha channel")
    return rows


def write_synthetic_openfake(root, rows, n_shards=3, rows_per_group=16, seed=0):
    """Write rows as Hugging-Face-style Parquet shards (`core/train-0000i-of-0000n.parquet`, several
    row groups each, rows shuffled across shards like the real dataset). Returns the shard paths."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    folder = Path(root) / "core"
    folder.mkdir(parents=True, exist_ok=True)
    order = np.random.default_rng(seed).permutation(len(rows))
    paths = []
    for i in range(n_shards):
        part = [rows[j] for j in order[i::n_shards]]
        path = folder / f"train-{i:05d}-of-{n_shards:05d}.parquet"
        pq.write_table(pa.Table.from_pylist(part), path, row_group_size=rows_per_group)
        paths.append(str(path))
    return paths


def write_synthetic_artifact(root, per_folder=12, seed=1):
    """An ArtiFact-shaped folder: one sub-folder per generator or real source, each with a
    `metadata.csv` (filename, image_path, target, category) and small JPEGs."""
    rng = np.random.default_rng(seed)
    root = Path(root)
    for name, target in (("stylegan2", 1), ("ddpm", 2), ("coco", 0), ("ffhq", 0)):
        (root / name / "images").mkdir(parents=True, exist_ok=True)
        with open(root / name / "metadata.csv", "w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(["filename", "image_path", "target", "category"])
            for i in range(per_folder):
                filename = f"img{i:05d}.jpg"
                synthetic_picture(rng, target != 0, (100, 100)).save(root / name / "images" / filename, quality=90)
                writer.writerow([filename, f"images/{filename}", target, "scene"])
    return str(root)


def write_synthetic_ffhq(root, n=16, seed=2):
    """An FFHQ-thumbnails-shaped folder: `n` 128x128 RGB PNGs of real 'faces'."""
    rng = np.random.default_rng(seed)
    folder = Path(root) / "thumbnails128x128"
    folder.mkdir(parents=True, exist_ok=True)
    for i in range(n):
        synthetic_picture(rng, False, (128, 128)).save(folder / f"{i:05d}.png")
    return str(root)


def save_json(path, value, secrets=None):
    """Write JSON with secrets scrubbed and numpy scalars converted."""
    Path(path).write_text(json.dumps(scrub_secrets(value, secrets), indent=2, default=_jsonable), encoding="utf-8")


def _jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (set, tuple)):
        return list(value)
    return str(value)

In [ ]:
# ---- inlined from engines/visual/metadata.py (edit the module, then rebuild) ----
"""The per-image metadata table (Task 1), generator families, and perceptual-hash dedupe (Task 2).

One row per image. Every leakage check, per-generator metric and error table reads this table.
Fields that are not known stay null and are counted (`null_counts`), never invented. The table
lists individual dataset images, so under the datasets' terms it is never committed to a public
repository (see `scripts/check_no_dataset_outputs.py`).
"""
import re

import numpy as np
import pandas as pd

METADATA_COLUMNS = [
    "image_id", "source_dataset", "generator", "generator_family", "is_fake", "manipulation_type",
    "source_image_id", "identity_id", "prompt_id", "width", "height", "format", "jpeg_quality",
    "file_size", "has_exif", "phash", "split", "license",
    # Not in the handbook's list, needed by this design:
    "release_date", "pipeline_file_size", "valid", "drop_reason", "dup_cluster", "group_id", "locator",
]
# `fold_role` is one column per fold, named `fold_role__<family>` (see splits.add_fold_roles).

# Licenses as found on each dataset's own page, with where and when they were read. A conflict is
# recorded as a conflict, not resolved here.
LICENSES = {
    "openfake": "cc-by-nc-4.0 (dataset card, read 2026-10-07); the handbook reports CC BY-NC-SA 4.0 in the paper "
                "(not re-verified)",
    "artifact": "not yet looked up (Task 0 records the license field of the Kaggle copy)",
    "ffhq": "Kaggle re-upload license field 'Unknown' (as seen by the user, 2026-10-06); source NVlabs/ffhq-dataset "
            "lists CC BY-NC-SA 4.0",
}

# Proposed generator -> family rules, first match wins. A family groups generators that share an
# architecture lineage, so holding a family out removes every close relative from training.
# These are a starting proposal written from the OpenFake dataset card; they are confirmed (or
# changed) against the real `model` values that Task 0 prints, and the outcome is logged in
# docs/design_decisions.md.
FAMILY_RULES = [
    (r"flux", "flux"),
    (r"^midjourney", "midjourney"),
    (r"^imagen", "imagen"),
    (r"nano-banana", "nano-banana"),
    (r"^gpt-image", "gpt-image"),
    (r"dall", "dall-e"),
    (r"^ideogram", "ideogram"),
    (r"^recraft", "recraft"),
    (r"^grok", "grok"),
    (r"^sdxl|juggernaut|realvis", "sdxl"),
    (r"^sd-3", "sd-3"),
    (r"^sd-[12]|realistic-vision|dreamshaper|epicdream", "sd-1x-2x"),
    (r"^stylegan", "stylegan"),
]


def family_of(generator):
    """(family, rule) for a generator name. `rule` is the pattern that matched, or `fallback` when
    none did -- then the family is the name with its trailing version removed, and the generator
    is listed for a human to check."""
    name = str(generator).strip().lower()
    for pattern, family in FAMILY_RULES:
        if re.search(pattern, name):
            return family, pattern
    stem = re.sub(r"[-_. ]?v?\d+(\.\d+)*[a-z]?$", "", name)
    return (stem or name), "fallback"


def empty_row():
    return {c: None for c in METADATA_COLUMNS}


def phash_hex(value):
    """A 64-bit hash as a 16-character hex string (None stays None)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    return value if isinstance(value, str) else f"{int(value):016x}"


def build_table(rows):
    """Rows (dicts) -> the metadata DataFrame with every schema column present.

    The 64-bit hash is stored as a 16-character hex string, converted *before* the rows become a
    DataFrame: a numeric column that also holds a missing value is turned into floats by pandas,
    and a float cannot hold 64 bits exactly, so the low bits of every hash would be lost.
    """
    rows = [{**empty_row(), **row, "phash": phash_hex(row.get("phash"))} for row in rows]
    table = pd.DataFrame(rows)
    for column in METADATA_COLUMNS:
        if column not in table.columns:
            table[column] = None
    return table


def null_counts(table, columns=METADATA_COLUMNS):
    """How many rows have no value, per column -- the 'unknown fields are counted' rule."""
    return {c: int(table[c].isna().sum()) for c in columns if c in table.columns}


# ---------------------------------------------------------------------------------------------------
# Near-duplicate search on 64-bit perceptual hashes
# ---------------------------------------------------------------------------------------------------
_POPCOUNT8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)


def popcount64(values):
    """Number of set bits in each uint64."""
    values = np.asarray(values, dtype=np.uint64)
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(values).astype(np.int64)
    return _POPCOUNT8[values.view(np.uint8).reshape(values.shape + (8,))].sum(axis=-1).astype(np.int64)


def hashes_to_uint64(hex_hashes):
    return np.array([int(h, 16) for h in hex_hashes], dtype=np.uint64)


def near_duplicate_pairs(hashes, max_distance, block=2048):
    """All pairs (i < j) of hashes within `max_distance` bits, with their distance.

    Exact search without comparing every pair: a 64-bit hash is cut into 8 bytes, and two hashes
    that differ in at most 7 bits must agree on at least one whole byte. So only hashes sharing a
    byte are compared. (Hence the limit of 7.)
    """
    if max_distance > 7:
        raise ValueError("near_duplicate_pairs is exact only up to a distance of 7")
    hashes = np.asarray(hashes, dtype=np.uint64)
    found = {}
    for band in range(8):
        keys = ((hashes >> np.uint64(8 * band)) & np.uint64(0xFF)).astype(np.int64)
        order = np.argsort(keys, kind="stable")
        bounds = np.flatnonzero(np.diff(keys[order])) + 1
        for members in np.split(order, bounds):
            if len(members) < 2:
                continue
            members = np.sort(members)
            values = hashes[members]
            for start in range(0, len(members), block):
                distance = popcount64(values[start:start + block, None] ^ values[None, :])
                rows, cols = np.nonzero(distance <= max_distance)
                keep = cols > rows + start
                for r, c in zip(rows[keep], cols[keep]):
                    found[(int(members[r + start]), int(members[c]))] = int(distance[r, c])
    if not found:
        return np.zeros((0, 3), dtype=np.int64)
    return np.array(sorted((i, j, d) for (i, j), d in found.items()), dtype=np.int64)


def cluster_pairs(n, pairs):
    """Union-find over index pairs -> a cluster label per item (the smallest index in its cluster)."""
    parent = np.arange(n)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i, j in pairs:
        a, b = find(int(i)), find(int(j))
        if a != b:
            parent[max(a, b)] = min(a, b)
    return np.array([find(i) for i in range(n)])


def distance_histogram(table, max_distance=7):
    """Number of image pairs at each hash distance 0..max_distance among valid rows -- the evidence
    the dedupe tolerance is chosen from (a jump in the count marks where unrelated images start to
    collide)."""
    valid = table[table["valid"] & table["phash"].notna()]
    pairs = near_duplicate_pairs(hashes_to_uint64(valid["phash"]), max_distance)
    counts = np.bincount(pairs[:, 2], minlength=max_distance + 1) if len(pairs) else np.zeros(max_distance + 1, int)
    return {int(d): int(c) for d, c in enumerate(counts)}


def duplicate_examples(table, max_distance=7, per_distance=4):
    """A few (image_id, image_id, distance) pairs at each distance, for looking at real duplicate
    pairs before choosing the tolerance."""
    valid = table[table["valid"] & table["phash"].notna()].reset_index(drop=True)
    pairs = near_duplicate_pairs(hashes_to_uint64(valid["phash"]), max_distance)
    out = []
    for d in range(max_distance + 1):
        for i, j, _ in pairs[pairs[:, 2] == d][:per_distance]:
            out.append((valid["image_id"][i], valid["image_id"][j], d))
    return out


def dedupe_table(table, tolerance):
    """Mark exact and near duplicates (hash distance <= tolerance) as dropped.

    Each cluster of duplicates keeps one image (the smallest image_id). A cluster that mixes real
    and fake images is dropped whole: the same picture carrying both labels has no usable label.
    Returns (table, report). Rows are marked (`valid` False, `drop_reason` 'duplicate'), not deleted,
    so the counts stay auditable.
    """
    table = table.copy()
    live = np.flatnonzero((table["valid"] & table["phash"].notna()).to_numpy())
    hashes = hashes_to_uint64(table["phash"].iloc[live])
    pairs = near_duplicate_pairs(hashes, tolerance)
    clusters = cluster_pairs(len(live), pairs[:, :2])
    table["dup_cluster"] = None
    report = {"tolerance": int(tolerance), "images_checked": int(len(live)), "pairs_within_tolerance": int(len(pairs)),
              "clusters": 0, "removed": 0, "mixed_label_clusters": 0, "cross_source_clusters": 0,
              "removed_by_source": {}}
    ids = table["image_id"].iloc[live].to_numpy()
    fake = table["is_fake"].iloc[live].to_numpy()
    source = table["source_dataset"].iloc[live].to_numpy()
    drop = []
    for label in np.unique(clusters):
        members = np.flatnonzero(clusters == label)
        if len(members) < 2:
            continue
        report["clusters"] += 1
        table.iloc[live[members], table.columns.get_loc("dup_cluster")] = str(ids[members].min())
        report["cross_source_clusters"] += int(len(set(source[members])) > 1)
        if len(set(fake[members])) > 1:
            report["mixed_label_clusters"] += 1
            drop.extend(members)
        else:
            keeper = members[np.argmin(ids[members])]
            drop.extend(m for m in members if m != keeper)
    rows = live[np.array(drop, dtype=int)] if drop else np.array([], dtype=int)
    table.iloc[rows, table.columns.get_loc("valid")] = False
    table.iloc[rows, table.columns.get_loc("drop_reason")] = "duplicate"
    report["removed"] = int(len(rows))
    report["removed_by_source"] = {str(k): int(v) for k, v in table.iloc[rows]["source_dataset"].value_counts().items()}
    return table, report


def cleaning_report(table):
    """Counts of what cleaning dropped and why, per source."""
    dropped = table[~table["valid"].astype(bool)]
    return {"rows": int(len(table)), "valid": int(table["valid"].astype(bool).sum()),
            "dropped_by_reason": {str(k): int(v) for k, v in dropped["drop_reason"].value_counts().items()},
            "dropped_by_source": {str(k): int(v) for k, v in dropped["source_dataset"].value_counts().items()}}

In [ ]:
# ---- inlined from engines/visual/scan.py (edit the module, then rebuild) ----
"""The two passes over the image sources.

Pass 1 (`scan_*`, Tasks 1-2) reads each selected image once to fill its metadata row: validity,
file properties, perceptual hash. No encoder, no GPU.
Pass 2 (`load_*`, Task 5) reads the images that survived cleaning and splitting again, to embed them.

Two passes because the shortcut baselines (Task 4) must pass on the metadata before any encoder
runs. Neither pass writes an image to disk; pass 1 keeps a metadata row, pass 2 an embedding.
"""
from concurrent.futures import ThreadPoolExecutor


REAL_FAMILY = "real"


def _image_fields(record):
    return {"width": record["width"], "height": record["height"], "format": record["format"],
            "jpeg_quality": record["jpeg_quality"], "file_size": record["file_size"], "has_exif": record["has_exif"],
            "phash": record["phash"], "pipeline_file_size": record["pipeline_file_size"],
            "valid": bool(record["ok"]), "drop_reason": record["reason"]}


def openfake_row(meta, locator):
    """The metadata fields of an OpenFake row that are known before its image is read."""
    fake = int(meta["label"] == "fake")
    image_id = f"openfake:{meta['row_id']}"
    return {
        "image_id": image_id, "source_dataset": "openfake", "generator": meta["model"],
        "generator_family": family_of(meta["model"])[0] if fake else REAL_FAMILY, "is_fake": fake,
        # OpenFake fakes are whole generated images (frames, for the video models); its reals are untouched photos.
        "manipulation_type": ("generated_video_frame" if meta.get("type") == "video" else "fully_generated") if fake else "none",
        "source_image_id": None if fake else image_id, "identity_id": None,
        "prompt_id": prompt_id_of(meta.get("prompt")), "license": LICENSES["openfake"],
        "release_date": meta.get("release_date"), "locator": locator,
    }


def keep_row(image_id, generator, keep_fraction, seed=0):
    """Deterministic thinning of over-represented generators: a row is kept when a hash of its id
    falls below that generator's fraction (default 1 = keep everything)."""
    fraction = keep_fraction.get(generator, keep_fraction.get("*", 1.0)) if keep_fraction else 1.0
    return fraction >= 1.0 or stable_fraction(image_id, seed) < fraction


def scan_openfake_shard(reader, shard, keep_fraction=None, seed=0, threads=4):
    """Pass 1 over one shard: metadata rows for every kept row. The small columns are read first;
    a row group's images are downloaded only if it holds a kept row."""
    sparse = SparseParquet(reader, shard["path"], shard["size"])
    columns = [c for c in META_COLUMNS if c in sparse.columns]
    rows, stats = [], {"rows_in_shard": int(sparse.meta.num_rows), "kept": 0, "row_groups_downloaded": 0}
    with ThreadPoolExecutor(max_workers=threads) as pool:
        for group in range(sparse.num_row_groups):
            meta = sparse.read(columns, [group]).to_pylist()
            wanted = [i for i, m in enumerate(meta) if keep_row(f"openfake:{m['row_id']}", m["model"], keep_fraction, seed)]
            if not wanted:
                continue
            blobs = image_bytes_column(sparse.read(["image"], [group]))
            stats["row_groups_downloaded"] += 1
            for i, record in zip(wanted, pool.map(inspect_image, [blobs[i] for i in wanted])):
                rows.append({**openfake_row(meta[i], f"{shard['path']}#{group}#{i}"), **_image_fields(record)})
    stats["kept"] = len(rows)
    return rows, stats


def scan_folder_rows(index_rows, threads=4):
    """Pass 1 over files on disk (ArtiFact, FFHQ): `index_rows` come from data.index_artifact /
    data.index_ffhq and carry `path`."""
    def one(row):
        record = inspect_image(read_file_bytes(row["path"]))
        fake = int(row["is_fake"])
        out = {"image_id": row["image_id"], "source_dataset": row["source_dataset"], "generator": row["generator"],
               "generator_family": family_of(row["generator"])[0] if fake else REAL_FAMILY, "is_fake": fake,
               "manipulation_type": None if fake else "none",
               "source_image_id": row.get("source_image_id") or (None if fake else row["image_id"]),
               "identity_id": row.get("identity_id"), "prompt_id": None,
               "license": LICENSES.get(row["source_dataset"]), "release_date": None, "locator": row["path"]}
        return {**out, **_image_fields(record)}

    with ThreadPoolExecutor(max_workers=threads) as pool:
        return list(pool.map(one, index_rows))


def load_openfake_items(reader, shard, variants_by_locator):
    """Pass 2 for one shard: (image_id, image bytes, variants) for the rows named in
    `variants_by_locator` ({locator: (image_id, [variants])}). Only the row groups that hold a
    wanted row are downloaded."""
    by_group = {}
    for locator, (image_id, variants) in variants_by_locator.items():
        _, group, row = locator.rsplit("#", 2)
        by_group.setdefault(int(group), []).append((int(row), image_id, variants))
    sparse = SparseParquet(reader, shard["path"], shard["size"])
    items = []
    for group in sorted(by_group):
        blobs = image_bytes_column(sparse.read(["image"], [group]))
        items.extend((image_id, blobs[row], variants) for row, image_id, variants in sorted(by_group[group]))
    return items


def load_folder_items(rows):
    """Pass 2 for files on disk: `rows` are (image_id, path, variants)."""
    return [(image_id, read_file_bytes(path), variants) for image_id, path, variants in rows]


def fetch_image_bytes(rows, reader=None, shard_sizes=None):
    """{image_id: bytes} for a handful of rows given as (image_id, source_dataset, locator) --
    used to look at duplicate pairs and to load the capped pixel subset of the heavier baselines.
    The bytes stay in memory."""
    out, by_shard = {}, {}
    for image_id, source, locator in rows:
        if source == "openfake":
            by_shard.setdefault(locator.rsplit("#", 2)[0], {})[locator] = (image_id, ["clean"])
        else:
            out[image_id] = read_file_bytes(locator)
    for path, wanted in by_shard.items():
        for image_id, data, _ in load_openfake_items(reader, {"path": path, "size": shard_sizes[path]}, wanted):
            out[image_id] = data
    return out

In [ ]:
# ---- inlined from engines/visual/splits.py (edit the module, then rebuild) ----
"""Group-aware splits with rotating held-out generator families (Task 3; design from
docs/HANDBOOK_ADDENDUM_01_ROTATING_HOLDOUT.md).

Two layers:

1. **Partition, fixed once.** Images are tied into groups by shared `source_image_id`,
   `identity_id` or `prompt_id` (in OpenFake a real photo and the fakes generated from its caption
   share a prompt). Each *group* is placed whole into one of four partitions -- `train`, `val`,
   `known_test`, `holdout` -- by a hash of its key, so the placement is the same in every session.
   The real images of the `holdout` partition are the reserved real pool.

2. **Fold roles, one set per held-out family.** In the fold for family F:
     train / val / known_test = those partitions without any fake of family F;
     unseen_test              = the fakes of family F in the holdout partition, plus an equal
                                number of reserved reals;
     everything else is unused in this fold (family-F fakes outside the holdout partition are
     never trained on; other families' holdout fakes wait for their own fold).

   Because the unseen test is drawn only from the holdout partition, no prompt, source image or
   identity is shared between a fold's training data and its unseen test, and the reserved reals
   are never in any fold's training data.

Embeddings are computed once per image; folds only change which rows the cheap head reads.
"""
import numpy as np
import pandas as pd


PARTITIONS = ("train", "val", "known_test", "holdout")
GROUP_KEYS = ("source_image_id", "identity_id", "prompt_id")
FOLD_PREFIX = "fold_role__"
OLD_ONLY = "old_only"


def group_ids(table, keys=GROUP_KEYS):
    """A group label per row: rows that share any non-null key value are one group (transitively).
    The label is the smallest image_id in the group, so it is stable across sessions."""
    n = len(table)
    pairs = []
    for key in keys:
        if key not in table.columns:
            continue
        values = table[key].to_numpy()
        first_seen = {}
        for i, value in enumerate(values):
            if value is None or (isinstance(value, float) and np.isnan(value)):
                continue
            token = (key, value)
            if token in first_seen:
                pairs.append((first_seen[token], i))
            else:
                first_seen[token] = i
    clusters = cluster_pairs(n, pairs)
    ids = table["image_id"].to_numpy()
    label_of = {}
    for cluster, image_id in zip(clusters, ids):
        if cluster not in label_of or image_id < label_of[cluster]:
            label_of[cluster] = image_id
    return np.array([label_of[c] for c in clusters], dtype=object)


def partition_of(group_key, fractions, seed=0):
    """The partition a group key hashes into. `fractions` maps partition -> share (summing to 1)."""
    u, edge = stable_fraction(group_key, seed), 0.0
    for name in PARTITIONS:
        edge += fractions[name]
        if u < edge:
            return name
    return PARTITIONS[-1]


def assign_partitions(table, fractions, seed=0):
    """Fill `group_id` and `split` for every valid row; invalid rows get no split."""
    table = table.copy()
    table["group_id"] = group_ids(table)
    valid = table["valid"].astype(bool).to_numpy()
    table["split"] = [partition_of(g, fractions, seed) if ok else None for g, ok in zip(table["group_id"], valid)]
    return table


def apply_generator_caps(table, caps, seed=0):
    """Keep at most `caps[partition]` images per (generator, partition); the rest are marked
    dropped with reason `over_cap`. Which images stay is decided by a hash of the image id, so it
    does not depend on row order. `caps` may hold `{'real': {...}, 'fake': {...}}` per-class caps."""
    table = table.copy()
    live = table[table["valid"].astype(bool)]
    order = pd.Series([stable_fraction(i, seed) for i in live["image_id"]], index=live.index)
    drop = []
    for (generator, split, is_fake), members in live.groupby(["generator", "split", "is_fake"]).groups.items():
        per_class = caps.get("fake" if is_fake else "real", caps)
        cap = per_class.get(split)
        if cap is not None and len(members) > cap:
            drop.extend(order.loc[members].sort_values().index[cap:])
    table.loc[drop, "valid"] = False
    table.loc[drop, "drop_reason"] = "over_cap"
    table.loc[drop, "split"] = None
    return table


def family_counts(table):
    """Valid fake images per family and partition."""
    fakes = table[table["valid"].astype(bool) & (table["is_fake"] == 1)]
    counts = fakes.groupby(["generator_family", "split"]).size().unstack(fill_value=0)
    return counts.reindex(columns=list(PARTITIONS), fill_value=0)


def eligible_families(table, min_unseen_fakes, min_reserved_reals=None):
    """Which families can be held out: a family needs at least `min_unseen_fakes` fakes in the
    holdout partition (its unseen test) and the reserved pool needs as many reals. Returns
    {family: {..counts.., 'eligible': bool, 'why_not': str or None}}."""
    counts = family_counts(table)
    live = table[table["valid"].astype(bool)]
    reserved = int(((live["is_fake"] == 0) & (live["split"] == "holdout")).sum())
    need_reals = min_unseen_fakes if min_reserved_reals is None else min_reserved_reals
    out = {}
    for family, row in counts.iterrows():
        why = None
        if row["holdout"] < min_unseen_fakes:
            why = f"only {int(row['holdout'])} holdout fakes (< {min_unseen_fakes})"
        elif reserved < need_reals:
            why = f"only {reserved} reserved reals (< {need_reals})"
        elif int(counts["train"].sum() - row["train"]) == 0:
            why = "no other family left to train on"
        out[str(family)] = {**{p: int(row[p]) for p in PARTITIONS}, "reserved_reals": reserved,
                            "eligible": why is None, "why_not": why}
    return out


def fold_roles(table, family, seed=0, reals_per_fake=1.0):
    """The role of every row in the fold that holds out `family` (see the module docstring)."""
    valid = table["valid"].astype(bool).to_numpy()
    split = table["split"].to_numpy()
    fake = (table["is_fake"] == 1).to_numpy()
    held = (table["generator_family"] == family).to_numpy() & fake
    roles = np.full(len(table), None, dtype=object)
    for name in ("train", "val", "known_test"):
        roles[valid & (split == name) & ~held] = name
    roles[valid & (split != "holdout") & held] = "unused"
    in_holdout = valid & (split == "holdout")
    roles[in_holdout & fake & ~held] = "unused"
    roles[in_holdout & held] = "unseen_test"
    reserved = np.flatnonzero(in_holdout & ~fake)
    roles[reserved] = "reserved_real"
    n_take = min(len(reserved), int(round(reals_per_fake * int((in_holdout & held).sum()))))
    ids = table["image_id"].to_numpy()
    order = sorted(reserved, key=lambda i: stable_fraction(ids[i], f"{seed}|{family}"))
    roles[np.array(order[:n_take], dtype=int)] = "unseen_test"
    return roles


def old_only_roles(table, old_families):
    """The old-only ablation: train and validate on real images plus fakes from `old_families`
    only. Its tests are the unseen-test sets of the LOFO folds of the modern families, so the
    ablation and the folds are scored on identical images."""
    valid = table["valid"].astype(bool).to_numpy()
    split = table["split"].to_numpy()
    fake = (table["is_fake"] == 1).to_numpy()
    old = table["generator_family"].isin(list(old_families)).to_numpy()
    roles = np.full(len(table), None, dtype=object)
    roles[valid] = "unused"
    for name in ("train", "val", "known_test"):
        roles[valid & (split == name) & (~fake | old)] = name
    roles[valid & (split == "holdout") & ~fake] = "reserved_real"
    return roles


def add_fold_roles(table, families, seed=0, old_families=None):
    """One `fold_role__<family>` column per held-out family, plus `fold_role__old_only`."""
    table = table.copy()
    for family in families:
        table[FOLD_PREFIX + str(family)] = fold_roles(table, family, seed)
    if old_families is not None:
        table[FOLD_PREFIX + OLD_ONLY] = old_only_roles(table, old_families)
    return table


def fold_names(table):
    return [c[len(FOLD_PREFIX):] for c in table.columns if c.startswith(FOLD_PREFIX)]


def class_balance(table, fold):
    """Real/fake counts per role, and per generator within each role, for one fold."""
    column = FOLD_PREFIX + fold
    used = table[table[column].notna() & (table[column] != "unused")]
    per_role = used.groupby([column, "is_fake"]).size().unstack(fill_value=0)
    per_generator = used.groupby([column, "generator"]).size()
    return {"per_role": {str(role): {("fake" if k else "real"): int(v) for k, v in row.items()}
                         for role, row in per_role.iterrows()},
            "per_generator": {f"{role}/{gen}": int(v) for (role, gen), v in per_generator.items()}}


def leakage_report(table, phash_tolerance=0, strict=True):
    """The per-fold leakage checks of the addendum, section 5. Returns a report; with `strict`
    raises AssertionError listing every violation.

      1. the held-out family has zero rows in that fold's train and validation;
      2. reserved reals (real images of the holdout partition) are in no training or validation
         split of any fold;
      3. no perceptual hash (within `phash_tolerance` bits), source_image_id, identity_id or
         prompt_id is shared between two different splits of a fold;
      5. class balance per split and per generator (returned for printing).
    (Check 4, the shortcut-baseline AUC, is reported by baselines.py.)
    """
    violations, report = [], {"folds": {}, "phash_tolerance": int(phash_tolerance)}
    valid = table["valid"].astype(bool)
    reserved = valid & (table["is_fake"] == 0) & (table["split"] == "holdout")
    for fold in fold_names(table):
        column = FOLD_PREFIX + fold
        roles = table[column]
        entry = {"balance": class_balance(table, fold)}
        if fold != OLD_ONLY:
            held = (table["generator_family"] == fold) & (table["is_fake"] == 1)
            entry["held_out_rows_in_train_val"] = int((held & roles.isin(["train", "val"])).sum())
            if entry["held_out_rows_in_train_val"]:
                violations.append(f"fold {fold}: {entry['held_out_rows_in_train_val']} held-out rows in train/val")
        entry["reserved_reals_in_train_val"] = int((reserved & roles.isin(["train", "val", "known_test"])).sum())
        if entry["reserved_reals_in_train_val"]:
            violations.append(f"fold {fold}: {entry['reserved_reals_in_train_val']} reserved reals outside the unseen test")
        used = table[roles.isin(["train", "val", "known_test", "unseen_test"])]
        entry["shared_keys"] = {}
        for key in ("source_image_id", "identity_id", "prompt_id", "group_id"):
            spread = used[used[key].notna()].groupby(key)[column].nunique()
            entry["shared_keys"][key] = int((spread > 1).sum())
            if entry["shared_keys"][key]:
                violations.append(f"fold {fold}: {entry['shared_keys'][key]} {key} values span more than one split")
        hashed = used[used["phash"].notna()].reset_index(drop=True)
        pairs = near_duplicate_pairs(hashes_to_uint64(hashed["phash"]), phash_tolerance) if len(hashed) else []
        split_of = hashed[column].to_numpy() if len(hashed) else np.array([])
        entry["phash_pairs_across_splits"] = int(sum(1 for i, j, _ in pairs if split_of[i] != split_of[j]))
        if entry["phash_pairs_across_splits"]:
            violations.append(f"fold {fold}: {entry['phash_pairs_across_splits']} near-duplicate pairs across splits")
        report["folds"][fold] = entry
    report["violations"] = violations
    if strict and violations:
        raise AssertionError("leakage checks failed: " + "; ".join(violations))
    return report


def balanced_subsample(labels, rng):
    """Indices that keep every example of the smaller class and an equal-sized random sample of
    the larger one. This is the one place a seed changes which training rows a head sees."""
    labels = np.asarray(labels)
    pos, neg = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
    n = min(len(pos), len(neg))
    if len(pos) == len(neg) or n == 0:
        return np.arange(len(labels))
    keep_pos = pos if len(pos) == n else rng.choice(pos, n, replace=False)
    keep_neg = neg if len(neg) == n else rng.choice(neg, n, replace=False)
    return np.sort(np.concatenate([keep_pos, keep_neg]))


def auc_ci_halfwidth(auc, n_pos, n_neg):
    """Approximate 95% half-width of a ROC-AUC confidence interval (Hanley & McNeil, 1982). Used
    only to judge, before any model exists, how many test images make a fold's CI usefully narrow."""
    if n_pos < 1 or n_neg < 1:
        return float("nan")
    q1, q2 = auc / (2 - auc), 2 * auc * auc / (1 + auc)
    variance = (auc * (1 - auc) + (n_pos - 1) * (q1 - auc * auc) + (n_neg - 1) * (q2 - auc * auc)) / (n_pos * n_neg)
    return float(1.96 * np.sqrt(max(variance, 0.0)))

In [ ]:
# ---- inlined from engines/visual/evaluate.py (edit the module, then rebuild) ----
"""Metrics and confidence intervals for the Visual Engine.

Convention: **fake is the positive class** (label 1) and a score is P(fake).
  FPR = real images flagged as fake;  FNR = fake images that got through.

The stratified bootstrap and the EER come from the Voice Engine's evaluate.py (the code fixed for
issue #16) rather than being written again.
"""
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score


FAKE_LABEL = 1
NOT_RUN = "not yet run"
SUITE_METRICS = ("roc_auc", "pr_auc", "eer", "fpr", "fnr", "ece")


def fast_roc_auc(scores, labels):
    """ROC-AUC from ranks (ties averaged). NaN when only one class is present."""
    labels = np.asarray(labels)
    pos = labels == FAKE_LABEL
    n_pos, n_neg = int(pos.sum()), int((~pos).sum())
    if n_pos == 0 or n_neg == 0:
        return float("nan")
    ranks = rankdata(np.asarray(scores, dtype=np.float64))
    return float((ranks[pos].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def expected_calibration_error(probs, labels, n_bins=15):
    """ECE: over equal-width confidence bins, the average gap between confidence and accuracy,
    weighted by how many predictions fall in each bin. Confidence is the probability of the
    predicted class, max(p, 1 - p)."""
    probs, labels = np.asarray(probs, dtype=np.float64), np.asarray(labels)
    if len(probs) == 0:
        return float("nan")
    confidence = np.maximum(probs, 1.0 - probs)
    correct = ((probs >= 0.5).astype(int) == labels).astype(np.float64)
    bins = np.minimum((confidence * n_bins).astype(int), n_bins - 1)
    total = 0.0
    for b in np.unique(bins):
        members = bins == b
        total += members.mean() * abs(correct[members].mean() - confidence[members].mean())
    return float(total)


def reliability_curve(probs, labels, n_bins=10):
    """Points for a reliability diagram: per bin of P(fake), the mean prediction, the observed
    fraction of fakes and the count."""
    probs, labels = np.asarray(probs, dtype=np.float64), np.asarray(labels)
    bins = np.minimum((probs * n_bins).astype(int), n_bins - 1)
    return [{"bin": int(b), "mean_predicted": float(probs[bins == b].mean()),
             "fraction_fake": float((labels[bins == b] == FAKE_LABEL).mean()), "count": int((bins == b).sum())}
            for b in np.unique(bins)]


def confusion_at(probs, labels, threshold):
    """Counts at `threshold` (predict fake when P(fake) >= threshold)."""
    probs, labels = np.asarray(probs), np.asarray(labels)
    flagged, fake = probs >= threshold, labels == FAKE_LABEL
    return {"tp": int((flagged & fake).sum()), "fn": int((~flagged & fake).sum()),
            "fp": int((flagged & ~fake).sum()), "tn": int((~flagged & ~fake).sum())}


def point_metrics(probs, labels, threshold):
    """Every suite metric as a plain number. A metric that needs both classes is NaN without them."""
    probs, labels = np.asarray(probs, dtype=np.float64), np.asarray(labels)
    c = confusion_at(probs, labels, threshold)
    both = (c["tp"] + c["fn"]) > 0 and (c["fp"] + c["tn"]) > 0
    return {
        "roc_auc": fast_roc_auc(probs, labels),
        "pr_auc": float(average_precision_score(labels == FAKE_LABEL, probs)) if both else float("nan"),
        "eer": float(compute_eer(probs, labels, FAKE_LABEL)["eer"]) if both else float("nan"),
        "fpr": c["fp"] / (c["fp"] + c["tn"]) if (c["fp"] + c["tn"]) else float("nan"),
        "fnr": c["fn"] / (c["fn"] + c["tp"]) if (c["fn"] + c["tp"]) else float("nan"),
        "ece": expected_calibration_error(probs, labels),
    }


def _resample_indices(labels, rng):
    """Stratified resample; falls back to a plain one when a class is missing (fake-only sets)."""
    labels = np.asarray(labels)
    if (labels == FAKE_LABEL).all() or (labels != FAKE_LABEL).all():
        return rng.choice(len(labels), len(labels))
    return _stratified_resample(labels, rng, FAKE_LABEL)


def bootstrap_metric_cis(probs, labels, threshold, n_resamples=1000, seed=0, alpha=0.05, metrics=SUITE_METRICS):
    """{metric: {value, ci_low, ci_high}} with percentile bootstrap CIs, resampling within each
    class so the class counts never change."""
    probs, labels = np.asarray(probs, dtype=np.float64), np.asarray(labels)
    point = point_metrics(probs, labels, threshold)
    rng = np.random.default_rng(seed)
    samples = {m: [] for m in metrics}
    for _ in range(n_resamples):
        idx = _resample_indices(labels, rng)
        values = point_metrics(probs[idx], labels[idx], threshold)
        for m in metrics:
            samples[m].append(values[m])
    out = {}
    for m in metrics:
        finite = np.array([v for v in samples[m] if np.isfinite(v)])
        low, high = (np.quantile(finite, [alpha / 2, 1 - alpha / 2]) if len(finite) else (np.nan, np.nan))
        out[m] = {"value": float(point[m]), "ci_low": float(low), "ci_high": float(high)}
    return out


def rate_ci(flags, n_resamples=1000, seed=0, alpha=0.05):
    """A proportion (for example the share of fakes detected) with a bootstrap CI."""
    flags = np.asarray(flags, dtype=np.float64)
    if len(flags) == 0:
        return {"value": float("nan"), "ci_low": float("nan"), "ci_high": float("nan"), "n": 0}
    rng = np.random.default_rng(seed)
    means = flags[rng.integers(0, len(flags), size=(n_resamples, len(flags)))].mean(axis=1)
    return {"value": float(flags.mean()), "ci_low": float(np.quantile(means, alpha / 2)),
            "ci_high": float(np.quantile(means, 1 - alpha / 2)), "n": int(len(flags))}


def per_generator_results(probs, labels, generators, threshold, n_resamples=1000, seed=0):
    """Per fake generator: detection rate at the threshold, and ROC-AUC of that generator's fakes
    against all the suite's real images. Per real source: the false-positive rate."""
    probs, labels, generators = np.asarray(probs, dtype=np.float64), np.asarray(labels), np.asarray(generators)
    real = labels != FAKE_LABEL
    out = {}
    for g in sorted(set(generators.tolist())):
        members = generators == g
        if labels[members].max() == FAKE_LABEL:
            subset = members | real
            auc = bootstrap_metric_cis(probs[subset], labels[subset], threshold, n_resamples, seed, metrics=("roc_auc",))
            out[str(g)] = {"kind": "fake", "n": int(members.sum()), "roc_auc_vs_all_reals": auc["roc_auc"],
                           "detection_rate": rate_ci(probs[members] >= threshold, n_resamples, seed)}
        else:
            out[str(g)] = {"kind": "real", "n": int(members.sum()),
                           "false_positive_rate": rate_ci(probs[members] >= threshold, n_resamples, seed)}
    return out


def evaluate_suite(probs, labels, threshold, generators=None, n_resamples=1000, seed=0):
    """Everything reported for one test suite: metrics with CIs, the confusion matrix at the
    operating threshold, class counts and per-generator results."""
    labels = np.asarray(labels)
    result = {"metrics": bootstrap_metric_cis(probs, labels, threshold, n_resamples, seed),
              "confusion": confusion_at(probs, labels, threshold), "threshold": float(threshold),
              "counts": {"fake": int((labels == FAKE_LABEL).sum()), "real": int((labels != FAKE_LABEL).sum())},
              "n_resamples": int(n_resamples)}
    if generators is not None:
        result["per_generator"] = per_generator_results(probs, labels, generators, threshold, n_resamples, seed)
    return result


def paired_metric_difference(probs_a, probs_b, labels, metric="roc_auc", threshold=0.5, n_resamples=1000, seed=0,
                             alpha=0.05):
    """CI of metric(a) - metric(b) for two models scored on the same images (each resample uses
    one set of indices for both). `excludes_zero` False means the two are a tie at this sample size."""
    a, b, labels = np.asarray(probs_a, dtype=np.float64), np.asarray(probs_b, dtype=np.float64), np.asarray(labels)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        idx = _resample_indices(labels, rng)
        diffs.append(point_metrics(a[idx], labels[idx], threshold)[metric] -
                     point_metrics(b[idx], labels[idx], threshold)[metric])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = point_metrics(a, labels, threshold)[metric] - point_metrics(b, labels, threshold)[metric]
    return {"metric": metric, "difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "verdict": "different" if (low > 0 or high < 0) else "tie"}


def results_template(models, suites):
    """The reporting grid: one row per model, one column per suite, every cell 'not yet run'
    until a real run fills it."""
    return {model: {suite: NOT_RUN for suite in suites} for model in models}


def mean_std(values):
    values = [v for v in values if v is not None and np.isfinite(v)]
    if not values:
        return {"mean": None, "std": None, "n": 0}
    return {"mean": float(np.mean(values)), "std": float(np.std(values)) if len(values) > 1 else None, "n": len(values)}


def unpaired_metric_difference(probs_a, labels_a, probs_b, labels_b, metric="roc_auc", threshold=0.5,
                               n_resamples=1000, seed=0, alpha=0.05):
    """CI of metric(a) - metric(b) for two different test sets (each resampled on its own) -- the
    known-minus-unseen gap of a fold."""
    a, la = np.asarray(probs_a, dtype=np.float64), np.asarray(labels_a)
    b, lb = np.asarray(probs_b, dtype=np.float64), np.asarray(labels_b)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        ia, ib = _resample_indices(la, rng), _resample_indices(lb, rng)
        diffs.append(point_metrics(a[ia], la[ia], threshold)[metric] - point_metrics(b[ib], lb[ib], threshold)[metric])
    diffs = [d for d in diffs if np.isfinite(d)]
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2]) if diffs else (np.nan, np.nan)
    point = point_metrics(a, la, threshold)[metric] - point_metrics(b, lb, threshold)[metric]
    return {"metric": metric, "difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0)}

In [ ]:
# ---- inlined from engines/visual/calibrate.py (edit the module, then rebuild) ----
"""Calibration, the operating threshold and the uncertain band (Task 7).

All three are fitted on a fold's validation split only, which holds seen generators only. Whether
that calibration still holds on an unseen family is measured (ECE per suite), not assumed.

For TrustMesh's KYC screening a missed fake costs more than a false alarm, so the operating
threshold is the one that catches a target share of validation fakes, and anything inside the
uncertain band goes to a person. The visual score never rejects anyone on its own.
"""
import numpy as np
from scipy.optimize import minimize_scalar


def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-np.abs(x))), np.exp(-np.abs(x)) / (1.0 + np.exp(-np.abs(x))))


def binary_nll(logits, labels):
    """Mean negative log-likelihood of binary labels under sigmoid(logits)."""
    logits, labels = np.asarray(logits, dtype=np.float64), np.asarray(labels, dtype=np.float64)
    return float(np.mean(np.logaddexp(0.0, logits) - labels * logits))


def fit_temperature(logits, labels, bounds=(0.05, 20.0)):
    """The single number T > 0 that minimizes the NLL of sigmoid(logits / T) on validation data.
    T > 1 means the model was over-confident. Ranking (ROC-AUC) is unchanged by it."""
    logits, labels = np.asarray(logits, dtype=np.float64), np.asarray(labels)
    if len(set(labels.tolist())) < 2:
        return 1.0
    result = minimize_scalar(lambda log_t: binary_nll(logits / np.exp(log_t), labels),
                             bounds=(np.log(bounds[0]), np.log(bounds[1])), method="bounded")
    return float(np.exp(result.x))


def calibrated_probs(logits, temperature):
    return sigmoid(np.asarray(logits, dtype=np.float64) / temperature)


def threshold_for_recall(probs, labels, target_recall):
    """The highest threshold at which at least `target_recall` of the fakes are still flagged."""
    fakes = np.sort(np.asarray(probs, dtype=np.float64)[np.asarray(labels) == 1])
    if len(fakes) == 0:
        return 0.5
    allowed_misses = int(np.floor((1.0 - target_recall) * len(fakes) + 1e-9))
    return float(fakes[allowed_misses])


def threshold_for_fpr(probs, labels, target_fpr):
    """The lowest threshold at which at most `target_fpr` of the real images are flagged."""
    reals = np.sort(np.asarray(probs, dtype=np.float64)[np.asarray(labels) == 0])
    if len(reals) == 0:
        return 0.5
    allowed = int(np.floor(target_fpr * len(reals) + 1e-9))
    return float(np.nextafter(reals[len(reals) - 1 - allowed], np.inf))


def choose_operating_point(probs, labels, target_recall=0.95, band_recall=0.99, band_fpr=0.01):
    """Operating threshold and uncertain band from validation predictions.

      threshold : catches `target_recall` of validation fakes (the KYC setting: favour recall);
      band low  : below it, at most 1 - `band_recall` of validation fakes are found ("likely real");
      band high : above it, at most `band_fpr` of validation reals are found ("likely AI");
      in between: "uncertain", sent to a person.

    If the model separates validation so well that low would exceed high, the band collapses onto
    the threshold. The targets are settings, logged with the result, not facts about the data.
    """
    probs, labels = np.asarray(probs, dtype=np.float64), np.asarray(labels)
    threshold = threshold_for_recall(probs, labels, target_recall)
    low = threshold_for_recall(probs, labels, band_recall)
    high = threshold_for_fpr(probs, labels, band_fpr)
    if low > high:
        low = high = threshold
    low, high = min(low, threshold), max(high, threshold)
    return {"threshold": float(threshold), "band_low": float(low), "band_high": float(high),
            "target_recall": target_recall, "band_recall": band_recall, "band_fpr": band_fpr,
            "validation_share_uncertain": float(((probs >= low) & (probs < high)).mean()) if len(probs) else None,
            "n_validation": int(len(probs))}


def verdicts(probs, band_low, band_high):
    """`likely_real`, `uncertain` or `likely_ai` per image."""
    probs = np.asarray(probs, dtype=np.float64)
    return np.where(probs < band_low, "likely_real", np.where(probs >= band_high, "likely_ai", "uncertain"))

In [ ]:
# ---- inlined from engines/visual/baselines.py (edit the module, then rebuild) ----
"""Shortcut baselines (Task 4): can the label be guessed without looking at the picture?

A tiny classifier is trained on file properties only, and another on the source dataset only. If
either separates real from fake, a real model would use the same shortcut, so the pipeline has to
be fixed before any encoder runs.

The rule for "separates" is the confidence interval, not a cut-off someone picked: the baseline
passes only if the 95% bootstrap CI of its ROC-AUC includes 0.5 (same reasoning as issue #16).

Three feature sets:
  raw      -- the properties of each file as received (width, height, aspect ratio, format, JPEG
              quality, file size, EXIF present). Expected to fail: that is why the shared pipeline exists.
  pipeline -- the same seven properties after the shared pipeline. All of them are then identical
              for every image except the byte count of the re-encoded JPEG, so this measures what
              shortcut is left for the model.
  source   -- the source dataset alone.
"""
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


SHORTCUT_COLUMNS = ("width", "height", "aspect_ratio", "format", "jpeg_quality", "file_size", "has_exif")
SHORTCUT_MODES = ("raw", "pipeline", "source")


def shortcut_matrix(table, mode, categories=None):
    """(X, feature names, categories) for one feature set. `categories` fixes the one-hot columns
    so train and test matrices line up."""
    n = len(table)
    if mode == "source":
        values = table["source_dataset"].astype(str).to_numpy()
        categories = sorted(set(values)) if categories is None else categories
        return (np.array([[float(v == c) for c in categories] for v in values]).reshape(n, len(categories)),
                [f"source={c}" for c in categories], categories)
    if mode == "pipeline":
        width = height = np.full(n, float(CLIP_SIZE))
        formats = np.array(["JPEG"] * n)
        quality = np.full(n, float(STANDARD_JPEG_QUALITY))
        size = table["pipeline_file_size"].astype(float).to_numpy()
        exif = np.zeros(n)
    elif mode == "raw":
        width, height = table["width"].astype(float).to_numpy(), table["height"].astype(float).to_numpy()
        formats = table["format"].astype(str).to_numpy()
        quality = table["jpeg_quality"].astype(float).fillna(-1.0).to_numpy()
        size = table["file_size"].astype(float).to_numpy()
        exif = table["has_exif"].astype(float).fillna(0.0).to_numpy()
    else:
        raise ValueError(f"unknown shortcut feature set: {mode}")
    categories = sorted(set(formats)) if categories is None else categories
    columns = [width, height, width / np.maximum(height, 1.0), quality, np.log1p(size), exif]
    names = ["width", "height", "aspect_ratio", "jpeg_quality", "log_file_size", "has_exif"]
    for c in categories:
        columns.append((formats == c).astype(float))
        names.append(f"format={c}")
    return np.column_stack(columns), names, categories


def _fit_predict(kind, x_train, y_train, x_test, seed):
    if kind == "xgboost":
        from xgboost import XGBClassifier

        model = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, subsample=0.8,
                              random_state=seed, n_jobs=2, eval_metric="logloss")
    else:
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    model.fit(x_train, y_train)
    return model.predict_proba(x_test)[:, 1]


def shortcut_baseline(table, mode, kind="logreg", train_split="train", test_split="known_test",
                      n_resamples=1000, seed=0):
    """Train on one partition, score another, and report ROC-AUC with a stratified bootstrap CI.
    `separable` is True when the CI excludes 0.5 in either direction."""
    live = table[table["valid"].astype(bool)]
    train, test = live[live["split"] == train_split], live[live["split"] == test_split]
    result = {"mode": mode, "kind": kind, "n_train": int(len(train)), "n_test": int(len(test)),
              "auc": None, "ci_low": None, "ci_high": None, "separable": None}
    y_train, y_test = train["is_fake"].astype(int).to_numpy(), test["is_fake"].astype(int).to_numpy()
    if len(set(y_train.tolist())) < 2 or len(set(y_test.tolist())) < 2:
        result["note"] = "a split has only one class; the baseline cannot be trained or scored"
        return result
    x_train, names, categories = shortcut_matrix(train, mode)
    x_test, _, _ = shortcut_matrix(test, mode, categories)
    scores = _fit_predict(kind, x_train, y_train, x_test, seed)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = _resample_indices(y_test, rng)
        samples.append(fast_roc_auc(scores[idx], y_test[idx]))
    low, high = np.quantile(samples, [0.025, 0.975])
    result.update(auc=fast_roc_auc(scores, y_test), ci_low=float(low), ci_high=float(high),
                  separable=bool(low > 0.5 or high < 0.5), features=names, n_resamples=int(n_resamples))
    return result


def run_shortcut_baselines(table, n_resamples=1000, seed=0, kinds=("logreg", "xgboost")):
    """Every feature set with every model kind, plus the gate. The gate looks at what the model
    will actually receive -- the pipeline and source feature sets -- and passes only if neither
    is separable. The raw result is reported next to it as the size of the shortcut the pipeline
    removes."""
    results = [shortcut_baseline(table, mode, kind, n_resamples=n_resamples, seed=seed)
               for mode in SHORTCUT_MODES for kind in kinds]
    gated = [r for r in results if r["mode"] in ("pipeline", "source")]
    undecided = [r for r in gated if r["separable"] is None]
    failing = [r for r in gated if r["separable"]]
    gate = {"passed": not failing and not undecided,
            "failing": [f"{r['mode']}/{r['kind']} AUC {r['auc']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]"
                        for r in failing],
            "undecided": [f"{r['mode']}/{r['kind']}: {r.get('note')}" for r in undecided],
            "rule": "pass only if the 95% bootstrap CI of ROC-AUC includes 0.5 for every pipeline and source baseline"}
    return {"results": results, "gate": gate}


def source_class_table(table):
    """Per source dataset: real and fake counts, and the most common sizes and formats -- the
    source-versus-label picture the pre-flight asks for."""
    live = table[table["valid"].astype(bool)]
    out = {}
    for source, rows in live.groupby("source_dataset"):
        sizes = (rows["width"].astype(int).astype(str) + "x" + rows["height"].astype(int).astype(str)).value_counts()
        out[str(source)] = {"real": int((rows["is_fake"] == 0).sum()), "fake": int((rows["is_fake"] == 1).sum()),
                            "top_sizes": {str(k): int(v) for k, v in sizes.head(5).items()},
                            "formats": {str(k): int(v) for k, v in rows["format"].value_counts().items()}}
    return out

In [ ]:
# ---- inlined from engines/visual/embed.py (edit the module, then rebuild) ----
"""Frozen CLIP image encoder and the resumable embedding cache (Task 5).

The encoder is frozen (eval mode, no gradients). Each image is embedded once; only the embedding
and the image id are kept -- never the image. Embeddings are written in chunks, one file per chunk,
so a session that stops can resume, and each chunk prints a progress line (issue #27: a silent long
run looks the same as a hang).
"""
import hashlib
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch


CLIP_MODEL = "openai/clip-vit-large-patch14"


class ClipEncoder:
    """CLIP's image tower with its projection head. `embed` returns the projected image embedding
    (the vector CLIP compares with text; 768 numbers for ViT-L/14 according to its config -- Task 0
    confirms the real figure), not L2-normalized."""

    def __init__(self, name=None, device=torch.device("cpu"), fp16=True, model=None):
        if model is None:
            from transformers import CLIPVisionModelWithProjection

            name = name or CLIP_MODEL
            model = CLIPVisionModelWithProjection.from_pretrained(name)
        self.device = device
        self.half = bool(fp16) and device.type == "cuda"
        self.model = model.to(device).eval()
        if self.half:
            self.model.half()
        for param in self.model.parameters():
            param.requires_grad_(False)
        self.name = name or type(model).__name__
        self.dim = int(self.model.config.projection_dim)
        self.pre_projection_dim = int(self.model.config.hidden_size)

    @torch.no_grad()
    def embed(self, pixels):
        """[batch, 3, 224, 224] float pixels -> float32 array [batch, dim]."""
        x = pixels.to(self.device)
        out = self.model(pixel_values=x.half() if self.half else x.float())
        return out.image_embeds.float().cpu().numpy()


def tiny_clip_for_smoke():
    """A randomly initialised, tiny CLIP image tower: the real transformers code path with no
    download. Never used for a reported number."""
    from transformers import CLIPVisionConfig, CLIPVisionModelWithProjection

    config = CLIPVisionConfig(hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=2,
                              image_size=224, patch_size=32, projection_dim=24)
    torch.manual_seed(0)
    return CLIPVisionModelWithProjection(config)


def embedding_is_deterministic(encoder, pixels):
    """Same input twice in eval mode -> the same embedding (max absolute difference returned)."""
    a, b = encoder.embed(pixels), encoder.embed(pixels)
    return {"identical": bool(np.array_equal(a, b)), "max_abs_difference": float(np.abs(a - b).max())}


class ChunkStore:
    """Embedding chunks on disk: `<name>-<key>.npz` holding image ids, variant labels and vectors.

    `resume_dirs` are extra folders to look in for chunks finished by an earlier session (on Kaggle,
    the previous version's output attached as an input). A chunk found anywhere counts as done.
    """

    def __init__(self, out_dir, name="emb", resume_dirs=()):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.name = name
        self.dirs = [self.out_dir] + [Path(d) for d in resume_dirs if d and Path(d).is_dir()]

    def _file(self, key):
        return f"{self.name}-{key}.npz"

    def find(self, key):
        for folder in self.dirs:
            if (folder / self._file(key)).exists():
                return folder / self._file(key)
        return None

    def has(self, key):
        return self.find(key) is not None

    def save(self, key, image_ids, variants, embeddings):
        tmp = self.out_dir / (self._file(key) + ".tmp")
        with open(tmp, "wb") as handle:
            np.savez(handle, image_id=np.asarray(image_ids, dtype=str), variant=np.asarray(variants, dtype=str),
                     embedding=np.asarray(embeddings, dtype=np.float32))
        os.replace(tmp, self.out_dir / self._file(key))

    def keys(self):
        prefix = self.name + "-"
        return sorted({p.name[len(prefix):-4] for d in self.dirs for p in d.glob(f"{prefix}*.npz")})

    def load(self):
        """Every chunk concatenated: (image ids, variants, embeddings)."""
        ids, variants, vectors = [], [], []
        for key in self.keys():
            with np.load(self.find(key)) as chunk:
                ids.append(chunk["image_id"])
                variants.append(chunk["variant"])
                vectors.append(chunk["embedding"])
        if not ids:
            return np.array([], dtype=str), np.array([], dtype=str), np.zeros((0, 0), dtype=np.float32)
        return np.concatenate(ids), np.concatenate(variants), np.concatenate(vectors)

    def size_gb(self):
        return sum(p.stat().st_size for p in self.out_dir.glob(f"{self.name}-*.npz")) / 1e9


def _prepare_item(item, seed):
    image_id, data, variants = item
    record = decode_image(data)
    if not record["ok"]:
        return []
    return [(image_id, v, preprocess_image(variant_image(record["rgb"], v, image_id, seed))) for v in variants]


def embed_items(encoder, items, batch_size=64, seed=0, decode_threads=4):
    """(image_id, image bytes, [variants]) items -> (ids, variants, embeddings). Images are decoded
    a batch at a time, so only one batch of pixels is ever in memory."""
    ids, variants, vectors = [], [], []
    with ThreadPoolExecutor(max_workers=decode_threads) as pool:
        for start in range(0, len(items), batch_size):
            prepared = [p for group in pool.map(lambda it: _prepare_item(it, seed), items[start:start + batch_size])
                        for p in group]
            for b in range(0, len(prepared), batch_size):
                part = prepared[b:b + batch_size]
                vectors.append(encoder.embed(torch.stack([p[2] for p in part])))
                ids.extend(p[0] for p in part)
                variants.extend(p[1] for p in part)
    dim = encoder.dim
    return ids, variants, (np.concatenate(vectors) if vectors else np.zeros((0, dim), dtype=np.float32))


def extract_embeddings(encoder, chunks, store, batch_size=64, seed=0, decode_threads=4, log=print):
    """Run `embed_items` over chunks and save each one. `chunks` yields (key, load) where `load()`
    returns that chunk's items; a chunk already in the store is skipped without loading it."""
    started, done, skipped, vectors = time.time(), 0, 0, 0
    for key, load in chunks:
        if store.has(key):
            skipped += 1
            log(f"  embeddings: chunk {key} already done, skipped")
            continue
        t0 = time.time()
        items = load()
        ids, variants, embeddings = embed_items(encoder, items, batch_size, seed, decode_threads)
        store.save(key, ids, variants, embeddings)
        done += 1
        vectors += len(ids)
        log(f"  embeddings: chunk {key}: {len(items)} images -> {len(ids)} vectors in {time.time() - t0:.0f}s "
            f"(total {vectors} vectors, {time.time() - started:.0f}s, cache {store.size_gb():.2f} GB)")
    return {"chunks_done": done, "chunks_skipped": skipped, "vectors_written": vectors,
            "seconds": time.time() - started, "cache_gb": store.size_gb()}


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
# ---- inlined from engines/visual/heads.py (edit the module, then rebuild) ----
"""The heads trained on cached CLIP embeddings (Task 6): logistic regression, and a one-hidden-layer
MLP for comparison.

Both read L2-normalized embeddings and output one logit per image, higher = more likely fake.
Weights are saved as plain arrays (`.npz`) and the file's SHA-256 is recorded with the results.
"""
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression


C_GRID = (0.01, 0.1, 1.0, 10.0, 100.0)


def l2_normalize(x):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


class LinearHead:
    kind = "logreg"

    def __init__(self, coef, intercept, c=None):
        self.coef = np.asarray(coef, dtype=np.float64).reshape(-1)
        self.intercept = float(intercept)
        self.c = c

    def logits(self, x):
        return l2_normalize(x).astype(np.float64) @ self.coef + self.intercept

    def arrays(self):
        return {"kind": np.array(self.kind), "coef": self.coef, "intercept": np.array(self.intercept),
                "c": np.array(np.nan if self.c is None else self.c)}

    def n_params(self):
        return int(self.coef.size + 1)


def fit_logreg_head(x_train, y_train, x_val, y_val, c_grid=C_GRID, max_iter=2000):
    """L2-regularized logistic regression (L-BFGS). `C` is the value with the best validation
    ROC-AUC; ties go to the smaller C (stronger regularization). Given its inputs the fit is
    deterministic -- a seed changes nothing here."""
    xt, xv = l2_normalize(x_train), l2_normalize(x_val)
    tried, best = {}, None
    for c in c_grid:
        model = LogisticRegression(C=c, max_iter=max_iter)
        model.fit(xt, y_train)
        auc = fast_roc_auc(model.decision_function(xv), y_val)
        tried[str(c)] = auc
        if best is None or auc > best[0] + 1e-9:
            best = (auc, c, model)
    head = LinearHead(best[2].coef_, best[2].intercept_[0], best[1])
    return head, {"validation_auc_by_c": tried, "chosen_c": best[1], "validation_auc": best[0]}


class _MLP(torch.nn.Module):
    def __init__(self, dim, hidden, dropout):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(dim, hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout),
                                       torch.nn.Linear(hidden, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPHead:
    kind = "mlp"

    def __init__(self, dim, hidden=256, dropout=0.2, state=None):
        self.dim, self.hidden, self.dropout = int(dim), int(hidden), float(dropout)
        self.module = _MLP(self.dim, self.hidden, self.dropout)
        if state is not None:
            self.module.load_state_dict({k: torch.as_tensor(v) for k, v in state.items()})
        self.module.eval()

    @torch.no_grad()
    def logits(self, x):
        self.module.eval().cpu()
        return self.module(torch.from_numpy(l2_normalize(x))).numpy().astype(np.float64)

    def arrays(self):
        out = {"kind": np.array(self.kind), "dim": np.array(self.dim), "hidden": np.array(self.hidden),
               "dropout": np.array(self.dropout)}
        out.update({f"state/{k}": v.detach().cpu().numpy() for k, v in self.module.state_dict().items()})
        return out

    def n_params(self):
        return int(sum(p.numel() for p in self.module.parameters()))


def fit_mlp_head(x_train, y_train, x_val, y_val, seed=0, hidden=256, dropout=0.2, epochs=60, patience=6,
                 lr=1e-3, weight_decay=1e-4, batch_size=256, device=torch.device("cpu")):
    """One hidden layer, Adam, early stopping on validation ROC-AUC. The seed sets the initial
    weights and the mini-batch order."""
    torch.manual_seed(seed)
    head = MLPHead(x_train.shape[1], hidden, dropout)
    module = head.module.to(device)
    xt = torch.from_numpy(l2_normalize(x_train)).to(device)
    yt = torch.as_tensor(np.asarray(y_train), dtype=torch.float32, device=device)
    xv = torch.from_numpy(l2_normalize(x_val)).to(device)
    optimizer = torch.optim.Adam(module.parameters(), lr=lr, weight_decay=weight_decay)
    generator = torch.Generator().manual_seed(seed)
    best = {"auc": -1.0, "epoch": 0, "state": None}
    history, since = [], 0
    for epoch in range(1, epochs + 1):
        module.train()
        order = torch.randperm(len(xt), generator=generator).to(device)
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            optimizer.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(module(xt[idx]), yt[idx])
            loss.backward()
            optimizer.step()
        module.eval()
        with torch.no_grad():
            auc = fast_roc_auc(module(xv).cpu().numpy(), y_val)
        history.append(auc)
        if auc > best["auc"] + 1e-6:
            best = {"auc": auc, "epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in module.state_dict().items()}}
            since = 0
        else:
            since += 1
            if since >= patience:
                break
    if best["state"] is not None:
        module.load_state_dict(best["state"])
    module.cpu().eval()
    return head, {"validation_auc": best["auc"], "best_epoch": best["epoch"], "epochs_run": len(history), "seed": seed}


def save_head(path, head):
    """Write the head's arrays to `path` (.npz) and return the file's SHA-256."""
    with open(path, "wb") as handle:
        np.savez(handle, **head.arrays())
    return file_sha256(path)


def load_head(path):
    with np.load(path) as data:
        kind = str(data["kind"])
        if kind == "logreg":
            c = float(data["c"])
            return LinearHead(data["coef"], float(data["intercept"]), None if np.isnan(c) else c)
        state = {k[len("state/"):]: data[k] for k in data.files if k.startswith("state/")}
        return MLPHead(int(data["dim"]), int(data["hidden"]), float(data["dropout"]), state)

In [ ]:
# ---- inlined from engines/visual/experiment.py (edit the module, then rebuild) ----
"""One fold, end to end: train a head on cached embeddings, calibrate it on that fold's validation
split, pick the operating point there, then score each test suite once (Tasks 6-8).

Validation holds only generators seen in the fold, so the temperature, the threshold and the
uncertain band never see the held-out family.
"""
import numpy as np
import pandas as pd


JPEG_SUITES = ("jpeg90", "jpeg70", "jpeg50")


class EmbeddingBank:
    """Cached embeddings addressed by (image id, variant)."""

    def __init__(self, image_ids, variants, embeddings):
        self.embeddings = np.asarray(embeddings, dtype=np.float32)
        self.index = {(str(i), str(v)): k for k, (i, v) in enumerate(zip(image_ids, variants))}
        self.dim = int(self.embeddings.shape[1]) if self.embeddings.ndim == 2 else 0

    def variants(self):
        return sorted({v for _, v in self.index})

    def lookup(self, image_ids, variant):
        """(matrix, found mask) for `image_ids` under one variant; images without that embedding
        (for example a file that failed to decode in pass 2) are left out and counted by the caller."""
        rows = [self.index.get((str(i), variant)) for i in image_ids]
        found = np.array([r is not None for r in rows], dtype=bool)
        return self.embeddings[[r for r in rows if r is not None]], found


def fold_rows(table, fold, role):
    return table[table[FOLD_PREFIX + fold] == role]


def train_matrix(bank, rows, n_aug):
    """Training matrix: each image's clean embedding plus its cached augmented views."""
    xs, ys = [], []
    labels = rows["is_fake"].astype(int).to_numpy()
    for variant in ["clean"] + [f"aug{k}" for k in range(n_aug)]:
        x, found = bank.lookup(rows["image_id"], variant)
        xs.append(x)
        ys.append(labels[found])
    return np.concatenate(xs), np.concatenate(ys)


def suite_frame(bank, rows, variant, head, temperature):
    """Per-image predictions of one suite: image id, label, generator, raw and calibrated P(fake)."""
    x, found = bank.lookup(rows["image_id"], variant)
    kept = rows[found]
    logits = head.logits(x) if len(x) else np.zeros(0)
    return pd.DataFrame({"image_id": kept["image_id"].to_numpy(), "is_fake": kept["is_fake"].astype(int).to_numpy(),
                         "generator": kept["generator"].to_numpy(), "prob_raw": sigmoid(logits),
                         "prob": calibrated_probs(logits, temperature)}), int((~found).sum())


def what_a_seed_changes(kind, subsampled):
    """Said in words, because a seed that changes nothing must not be reported as variance."""
    parts = []
    if subsampled:
        parts.append("which images of the larger class are kept when the training set is balanced")
    if kind == "mlp":
        parts.append("the MLP's initial weights and mini-batch order")
    return "; ".join(parts) if parts else "nothing: this fit is deterministic given its inputs, so one run is reported"


def run_fold(table, bank, fold, kind="logreg", seed=0, n_aug=2, n_resamples=1000, with_ci=True,
             target_recall=0.95, band_recall=0.99, band_fpr=0.01, device=None, extra_tests=None):
    """Train, calibrate and score one fold. Returns (result, {"frames": {suite: per-image predictions}, "head": the fitted head}).

    `extra_tests` adds named test sets ({name: rows}) scored like the unseen test -- the old-only
    ablation uses it to be scored on every modern family's unseen-test images.
    """
    train, val = fold_rows(table, fold, "train"), fold_rows(table, fold, "val")
    rng = np.random.default_rng(seed)
    keep = balanced_subsample(train["is_fake"].astype(int).to_numpy(), rng)
    subsampled = len(keep) < len(train)
    train = train.iloc[keep]
    x_train, y_train = train_matrix(bank, train, n_aug)
    x_val, val_found = bank.lookup(val["image_id"], "clean")
    y_val = val["is_fake"].astype(int).to_numpy()[val_found]
    result = {"fold": fold, "kind": kind, "seed": seed, "n_aug_views": n_aug,
              "seed_changes": what_a_seed_changes(kind, subsampled),
              "counts": {"train_images": int(len(train)), "train_vectors": int(len(y_train)),
                         "train_fake_images": int((train["is_fake"] == 1).sum()),
                         "val": int(len(y_val)), "val_fake": int(y_val.sum())}}
    if len(set(y_train.tolist())) < 2 or len(set(y_val.tolist())) < 2:
        result["not_run"] = "training or validation split has a single class"
        return result, {"frames": {}, "head": None}
    if kind == "mlp":
        import torch

        head, fit = fit_mlp_head(x_train, y_train, x_val, y_val, seed=seed, device=device or torch.device("cpu"))
    else:
        head, fit = fit_logreg_head(x_train, y_train, x_val, y_val)
    val_logits = head.logits(x_val)
    temperature = fit_temperature(val_logits, y_val)
    val_probs = calibrated_probs(val_logits, temperature)
    operating = choose_operating_point(val_probs, y_val, target_recall, band_recall, band_fpr)
    result.update(fit=fit, temperature=temperature, operating_point=operating, n_params=head.n_params(),
                  validation={"ece_before": expected_calibration_error(sigmoid(val_logits), y_val),
                              "ece_after": expected_calibration_error(val_probs, y_val),
                              "roc_auc": point_metrics(val_probs, y_val, operating["threshold"])["roc_auc"]})

    tests = {"known": fold_rows(table, fold, "known_test"), "unseen": fold_rows(table, fold, "unseen_test")}
    tests.update(extra_tests or {})
    suites, frames, missing = {}, {}, {}
    for test_name, rows in tests.items():
        if len(rows) == 0:
            continue
        for variant in ("clean",) + DEGRADATIONS:
            name = test_name if variant == "clean" else f"{test_name}/{variant}"
            frame, lost = suite_frame(bank, rows, variant, head, temperature)
            if len(frame) == 0:
                continue
            labels, probs = frame["is_fake"].to_numpy(), frame["prob"].to_numpy()
            if with_ci:
                suite = evaluate_suite(probs, labels, operating["threshold"],
                                       frame["generator"].to_numpy() if variant == "clean" else None, n_resamples, seed)
            else:
                suite = {"metrics": {k: {"value": v} for k, v in point_metrics(probs, labels, operating["threshold"]).items()},
                         "counts": {"fake": int(labels.sum()), "real": int((labels == 0).sum())}}
            suite["ece_before_calibration"] = expected_calibration_error(frame["prob_raw"].to_numpy(), labels)
            suite["share_uncertain"] = float(((probs >= operating["band_low"]) & (probs < operating["band_high"])).mean())
            suites[name] = suite
            frames[name] = frame
            missing[name] = lost
    result["suites"] = suites
    result["missing_embeddings"] = {k: v for k, v in missing.items() if v}
    if with_ci and "known" in frames and "unseen" in frames:
        k, u = frames["known"], frames["unseen"]
        result["known_minus_unseen"] = {
            metric: unpaired_metric_difference(k["prob"].to_numpy(), k["is_fake"].to_numpy(), u["prob"].to_numpy(),
                                               u["is_fake"].to_numpy(), metric, operating["threshold"], n_resamples, seed)
            for metric in ("roc_auc", "eer")}
    return result, {"frames": frames, "head": head}


def metric_value(result, suite, metric="roc_auc"):
    """A suite metric's point value from a fold result, or None when that suite was not scored."""
    try:
        return result["suites"][suite]["metrics"][metric]["value"]
    except (KeyError, TypeError):
        return None

In [ ]:
# ---- inlined from engines/visual/pixel_baselines.py (edit the module, then rebuild) ----
"""Heavier pixel baselines (Task 9): a small CNN from scratch and a fine-tuned ResNet-50.

**Off by default** (`RUN_SMALL_CNN`, `RUN_RESNET50` in the notebook). Unlike the heads, these train
on pixels, which brings back the disk and time risk that sank Voice Models E and F. So the images
they use are a capped subset held in memory as small uint8 arrays -- never written to disk -- and
they run only after Tasks 1-8 have real results and the user says go.
"""
import numpy as np
import torch



def pixel_array(data, side):
    """Image bytes -> uint8 [side, side, 3] after the shared pipeline, or None for an invalid file."""
    record = decode_image(data)
    if not record["ok"]:
        return None
    image = standardize(record["rgb"])[0]
    if side != image.size[0]:
        image = image.resize((side, side))
    return np.asarray(image, dtype=np.uint8)


class SmallCNN(torch.nn.Module):
    """Four conv blocks and a linear layer -- the 'small CNN from scratch' baseline."""

    def __init__(self, channels=(16, 32, 64, 96)):
        super().__init__()
        layers, previous = [], 3
        for c in channels:
            layers += [torch.nn.Conv2d(previous, c, 3, padding=1), torch.nn.BatchNorm2d(c), torch.nn.ReLU(),
                       torch.nn.MaxPool2d(2)]
            previous = c
        self.features = torch.nn.Sequential(*layers, torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten())
        self.head = torch.nn.Linear(previous, 1)

    def forward(self, x):
        return self.head(self.features(x)).squeeze(-1)


def build_resnet50(pretrained=True):
    """torchvision's ResNet-50 with a one-logit head. `pretrained` downloads the ImageNet weights
    (Kaggle, internet on); the smoke test passes False."""
    from torchvision.models import resnet50

    model = resnet50(weights="IMAGENET1K_V2" if pretrained else None)
    model.fc = torch.nn.Linear(model.fc.in_features, 1)
    return model


class _OneLogit(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x).squeeze(-1)


def _to_tensor(images, device):
    x = torch.from_numpy(np.ascontiguousarray(images)).to(device).float().div_(255.0).permute(0, 3, 1, 2)
    mean = torch.tensor(CLIP_MEAN, device=device).view(1, 3, 1, 1)
    std = torch.tensor(CLIP_STD, device=device).view(1, 3, 1, 1)
    return (x - mean) / std


@torch.no_grad()
def pixel_logits(model, images, device, batch_size=64):
    model.eval()
    out = [model(_to_tensor(images[i:i + batch_size], device)).float().cpu() for i in range(0, len(images), batch_size)]
    return torch.cat(out).numpy().astype(np.float64) if out else np.zeros(0)


def train_pixel_model(model, x_train, y_train, x_val, y_val, device, epochs=8, patience=3, lr=1e-3, batch_size=64,
                      seed=0, log=print):
    """Train a pixel model on in-memory uint8 images with early stopping on validation ROC-AUC.
    Prints one line per epoch."""
    torch.manual_seed(seed)
    model = (_OneLogit(model) if not isinstance(model, SmallCNN) else model).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    labels = torch.as_tensor(np.asarray(y_train), dtype=torch.float32)
    generator = torch.Generator().manual_seed(seed)
    best = {"auc": -1.0, "epoch": 0, "state": None}
    since = 0
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(x_train), generator=generator).numpy()
        for start in range(0, len(order), batch_size):
            idx = order[start:start + batch_size]
            if len(idx) < 2:
                continue
            optimizer.zero_grad()
            loss = torch.nn.functional.binary_cross_entropy_with_logits(
                model(_to_tensor(x_train[idx], device)), labels[idx].to(device))
            loss.backward()
            optimizer.step()
        auc = fast_roc_auc(pixel_logits(model, x_val, device, batch_size), y_val)
        log(f"    epoch {epoch}: validation ROC-AUC {auc:.4f}")
        if auc > best["auc"] + 1e-6:
            best = {"auc": auc, "epoch": epoch, "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}}
            since = 0
        else:
            since += 1
            if since >= patience:
                break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    return model, {"validation_auc": best["auc"], "best_epoch": best["epoch"], "seed": seed,
                   "n_params": int(sum(p.numel() for p in model.parameters()))}

## Task 0 -- pre-flight: report, do not assume

Each check below writes what it found to `preflight_task0.json` (aggregate numbers only -- no
per-image rows). A check that fails is recorded with its error and the rest still run.

In [ ]:
# --- 0a. Environment, disk, token (presence only) -----------------------------------------------------
def check_environment():
    import PIL
    import pyarrow
    import transformers
    import huggingface_hub

    info = {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
            "pyarrow": pyarrow.__version__, "huggingface_hub": huggingface_hub.__version__, "pillow": PIL.__version__,
            "numpy": np.__version__, "pandas": pd.__version__, "cpu_count": os.cpu_count(),
            "cuda": torch.cuda.is_available(), "disk": disk_report(), "kaggle_inputs": find_kaggle_inputs(),
            "resume_dirs": RESUME_DIRS}
    if torch.cuda.is_available():
        info["gpus"] = [{"name": torch.cuda.get_device_name(i),
                         "memory_gb": round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1)}
                        for i in range(torch.cuda.device_count())]
    try:
        import psutil

        info["ram_gb"] = round(psutil.virtual_memory().total / 1e9, 1)
    except ImportError:
        info["ram_gb"] = None
    print(json.dumps(info, indent=1))
    return info


preflight_step("environment", check_environment)

_token = read_hf_token()
if _token:
    SECRETS.append(_token)
save_preflight("hf_token_present", bool(_token))   # only the fact that it exists -- never its value
print("HF_TOKEN secret present:", bool(_token))

In [ ]:
# --- 0b. OpenFake: what the repository really holds ---------------------------------------------------
if SMOKE_TEST:
    import fsspec

    OPENFAKE_FS = fsspec.filesystem("file")
    OPENFAKE_ROOT = os.environ["TRUSTMESH_OPENFAKE_DIR"]
else:
    from huggingface_hub import HfFileSystem

    OPENFAKE_FS = HfFileSystem(token=_token)
    OPENFAKE_ROOT = f"datasets/{OPENFAKE_REPO}"
OPENFAKE_SHARDS, OPENFAKE_READER = [], None


def check_openfake_files():
    global OPENFAKE_SHARDS, OPENFAKE_READER
    OPENFAKE_SHARDS = list_parquet_shards(OPENFAKE_FS, OPENFAKE_ROOT)
    by_split = {}
    for s in OPENFAKE_SHARDS:
        entry = by_split.setdefault(f"{s['config']}/{s['split']}", {"shards": 0, "gb": 0.0})
        entry["shards"] += 1
        entry["gb"] += s["size"] / 1e9
    out = {"root": OPENFAKE_ROOT, "parquet_shards": len(OPENFAKE_SHARDS),
           "total_gb": round(sum(s["size"] for s in OPENFAKE_SHARDS) / 1e9, 1),
           "by_split": {k: {"shards": v["shards"], "gb": round(v["gb"], 1)} for k, v in sorted(by_split.items())}}
    try:
        card = OPENFAKE_FS.cat_file(f"{OPENFAKE_ROOT}/README.md").decode("utf-8", "replace")
        out["card_license"] = next((line.split(":", 1)[1].strip() for line in card.splitlines()
                                    if line.startswith("license:")), None)
    except Exception as exc:  # noqa: BLE001
        out["card_license"] = f"could not read the dataset card: {type(exc).__name__}"
    out["paper_license_per_handbook"] = "CC BY-NC-SA 4.0 (the handbook's reading of the paper; not re-verified here)"
    try:   # the repository ships a note on what its v2 repair changed; record it rather than guess
        out["v2_repair_summary"] = json.loads(OPENFAKE_FS.cat_file(f"{OPENFAKE_ROOT}/corrections/v2_repair_summary.json"))
    except Exception as exc:  # noqa: BLE001
        out["v2_repair_summary"] = f"not read: {type(exc).__name__}"
    out["other_files"] = sorted(p.rsplit("/", 1)[-1] for p in OPENFAKE_FS.find(OPENFAKE_ROOT) if not p.endswith(".parquet"))[:30]

    # Two ways to read byte ranges; time a footer-sized read with each and keep the faster.
    readers = {"fsspec": FsRangeReader(OPENFAKE_FS)}
    if not SMOKE_TEST:
        readers["http"] = HttpRangeReader(OPENFAKE_REPO, _token)
    timings = {}
    for name, reader in readers.items():
        try:
            t0 = time.time()
            for s in OPENFAKE_SHARDS[:3]:
                reader.read(s["path"], max(0, s["size"] - 65536), s["size"])
            timings[name] = round((time.time() - t0) / max(1, len(OPENFAKE_SHARDS[:3])), 3)
        except Exception as exc:  # noqa: BLE001
            timings[name] = f"failed: {type(exc).__name__}: {scrub_secrets(str(exc), SECRETS)[:200]}"
    usable = {k: v for k, v in timings.items() if isinstance(v, float)}
    # Prefer the plain HTTP reader when it works: it asks the Hub once per file and then reads from storage directly,
    # which keeps tens of thousands of small range reads clear of the Hub's request limits.
    chosen = "http" if "http" in usable else min(usable, key=usable.get)
    OPENFAKE_READER = readers[chosen]
    out["range_read_seconds_per_request"] = timings
    out["range_reader_used"] = chosen
    print(json.dumps(out, indent=1))
    return out


preflight_step("openfake_files", check_openfake_files)

In [ ]:
# --- 0c. OpenFake: real counts from the metadata columns only (no image is downloaded) ----------------
OPENFAKE_SCAN, OPENFAKE_COUNTS = None, None


def check_openfake_counts():
    global OPENFAKE_SCAN, OPENFAKE_COUNTS
    OPENFAKE_SCAN = scan_all_shards(OPENFAKE_READER, OPENFAKE_SHARDS, threads=SCAN_THREADS,
                                    max_seconds=None if SMOKE_TEST else 45 * 60, progress_every=0 if SMOKE_TEST else 25)
    # Parquet files without an image column (the repository's `corrections/` tables) are listed, not counted.
    other = [s for s in OPENFAKE_SCAN["shards"] if "image" not in s["columns"]]
    OPENFAKE_SCAN["shards"] = [s for s in OPENFAKE_SCAN["shards"] if "image" in s["columns"]]
    done = OPENFAKE_SCAN["shards"]
    rows = []
    for shard in done:
        for (label, model, kind, date), n in shard["counts"].items():
            rows.append({"config": shard["config"], "split": shard["split"], "shard": shard["path"].rsplit("/", 1)[-1],
                         "label": label, "model": model, "type": kind, "release_date": date, "rows": n})
    OPENFAKE_COUNTS = pd.DataFrame(rows)
    # Aggregate tables (counts per shard and per model -- not per-image rows).
    OPENFAKE_COUNTS.to_csv(OUTPUT_DIR / "openfake_shard_counts.csv", index=False)
    per_model = (OPENFAKE_COUNTS.groupby(["config", "split", "label", "model", "type", "release_date"], dropna=False)["rows"]
                 .sum().reset_index().sort_values(["config", "split", "label", "rows"], ascending=[True, True, True, False]))
    per_model.to_csv(OUTPUT_DIR / "openfake_model_counts.csv", index=False)

    group_rows = np.concatenate([s["row_group_rows"] for s in done]) if done else np.array([0])
    out = {
        "shards_scanned": len(done), "shards_failed": OPENFAKE_SCAN["failed"][:10],
        "shards_skipped_for_time": len(OPENFAKE_SCAN["skipped_for_time"]), "scan_seconds": round(OPENFAKE_SCAN["seconds"], 1),
        "metadata_bytes_read_mb": round(OPENFAKE_READER.bytes_read / 1e6, 1), "range_requests": OPENFAKE_READER.requests,
        "columns": done[0]["columns"] if done else None, "total_rows": int(sum(s["rows"] for s in done)),
        "rows_by_split": {f"{c}/{s}": int(n) for (c, s), n in OPENFAKE_COUNTS.groupby(["config", "split"])["rows"].sum().items()},
        "rows_by_split_and_label": {f"{c}/{s}/{l}": int(n) for (c, s, l), n in
                                    OPENFAKE_COUNTS.groupby(["config", "split", "label"])["rows"].sum().items()},
        "label_values": {str(k): int(v) for k, v in OPENFAKE_COUNTS.groupby("label")["rows"].sum().items()},
        "type_values": {str(k): int(v) for k, v in OPENFAKE_COUNTS.groupby("type", dropna=False)["rows"].sum().items()},
        "distinct_model_values": int(OPENFAKE_COUNTS["model"].nunique()),
        "distinct_fake_generators": int(OPENFAKE_COUNTS[OPENFAKE_COUNTS["label"] == "fake"]["model"].nunique()),
        "real_sources": {str(k): int(v) for k, v in
                         OPENFAKE_COUNTS[OPENFAKE_COUNTS["label"] == "real"].groupby("model")["rows"].sum().items()},
        "distinct_release_dates": int(OPENFAKE_COUNTS["release_date"].nunique()),
        "row_groups_per_shard_median": float(np.median([s["row_groups"] for s in done])) if done else None,
        "rows_per_row_group_quantiles": {q: float(np.quantile(group_rows, float(q))) for q in ("0.05", "0.5", "0.95")},
        "image_mb_per_row_by_split": {
            f"{c}/{sp}": round(sum(s["image_bytes"] for s in done if (s["config"], s["split"]) == (c, sp)) / 1e6 /
                               max(1, sum(s["rows"] for s in done if (s["config"], s["split"]) == (c, sp))), 3)
            for c, sp in sorted({(s["config"], s["split"]) for s in done})},
        "sparse_read_misses": int(sum(s["sparse_misses"] for s in done)),
        "hub_requests": getattr(OPENFAKE_READER, "hub_requests", None),
        "parquet_files_without_images": [{"file": "/".join(s["path"].rsplit("/", 2)[-2:]), "rows": s["rows"],
                                          "columns": s["columns"]} for s in other],
        "prompt_groups_all": prompt_group_stats(OPENFAKE_SCAN),
        "prompt_groups_core_train_validation": prompt_group_stats(
            {"shards": [s for s in done if s["config"] == "core" and s["split"] in ("train", "validation")]}),
    }
    # Fake generators with the proposed family and which rule produced it ("fallback" = needs a human look).
    fakes = OPENFAKE_COUNTS[OPENFAKE_COUNTS["label"] == "fake"]
    generators = []
    for model, part in fakes.groupby("model"):
        family, rule = family_of(model)
        generators.append({"model": model, "rows": int(part["rows"].sum()),
                           "rows_by_split": {f"{c}/{s}": int(n) for (c, s), n in part.groupby(["config", "split"])["rows"].sum().items()},
                           "types": sorted(set(part["type"].astype(str))), "release_dates": sorted(set(part["release_date"].astype(str))),
                           "proposed_family": family, "family_rule": rule})
    out["fake_generators"] = sorted(generators, key=lambda g: -g["rows"])
    families = Counter()
    for g in generators:
        families[g["proposed_family"]] += g["rows"]
    out["proposed_families"] = dict(families.most_common())
    out["generators_on_fallback_rule"] = [g["model"] for g in generators if g["family_rule"] == "fallback"]

    print(json.dumps({k: v for k, v in out.items() if k not in ("fake_generators",)}, indent=1))
    print(f"\n{'fake generator':38s} {'rows':>9s}  {'type':16s} {'released':10s} proposed family (rule)")
    for g in out["fake_generators"]:
        print(f"{g['model'][:38]:38s} {g['rows']:9d}  {','.join(g['types'])[:16]:16s} {','.join(g['release_dates'])[:10]:10s} "
              f"{g['proposed_family']} ({g['family_rule']})")
    return out


preflight_step("openfake_counts", check_openfake_counts)

In [ ]:
# --- 0d. OpenFake: a timed streamed sample (images downloaded, decoded, then dropped) -----------------
OPENFAKE_SAMPLE = None


def core_train_shards():
    return [s for s in OPENFAKE_SHARDS if s["config"] == "core" and s["split"] == "train"]


def seeded_shard_order(shards):
    """A fixed order for reading shards: the same in every session, unrelated to file names."""
    return sorted(shards, key=lambda s: stable_fraction(s["path"], SEED))


def check_streaming():
    global OPENFAKE_SAMPLE
    pool = core_train_shards()
    picks = seeded_shard_order(pool)[:2]
    OPENFAKE_READER.requests = OPENFAKE_READER.bytes_read = 0
    OPENFAKE_SAMPLE = time_streamed_sample(OPENFAKE_READER, picks, max_images=40 if SMOKE_TEST else 2400,
                                           max_seconds=60 if SMOKE_TEST else 900, decode_threads=THREADS)
    out = {k: v for k, v in OPENFAKE_SAMPLE.items() if k != "row_groups"}
    out["shards_sampled"] = [p["path"].rsplit("/", 1)[-1] for p in picks]
    out["row_groups_read"] = len(OPENFAKE_SAMPLE["row_groups"])
    out["seek_row_groups"] = [g for g in OPENFAKE_SAMPLE["row_groups"] if g["entered_by_seek"]]
    out["what_is_timed"] = ("download = one HTTP range per row group's image column; decode = decode + upright RGB + "
                            "perceptual hash + the shared resize/JPEG pipeline, on THREADS threads")
    print(json.dumps(out, indent=1))
    return out


preflight_step("streaming_sample", check_streaming)

In [ ]:
# --- 0e. Can per-generator quotas be filled, and how long would the planned sample take? --------------
def check_quota_feasibility():
    pool = seeded_shard_order(core_train_shards())
    scanned = {s["path"]: s for s in (OPENFAKE_SCAN or {"shards": []})["shards"]}
    ordered = [scanned[s["path"]] for s in pool if s["path"] in scanned]
    if not ordered or OPENFAKE_SAMPLE is None:
        raise RuntimeError("needs the metadata scan and the streamed sample above")
    per_shard_family, per_shard_model = [], []
    for shard in ordered:
        models = Counter(m for l, m in zip(shard["label"], shard["model"]) if l == "fake")
        per_shard_model.append(models)
        fam = Counter()
        for m, n in models.items():
            fam[family_of(m)[0]] += n
        per_shard_family.append(fam)
    total_family = sum(per_shard_family, Counter())
    mean_shard_gb = float(np.mean([s["size"] for s in ordered])) / 1e9
    mean_rows = float(np.mean([s["rows"] for s in ordered]))
    mbps = OPENFAKE_SAMPLE["mb_per_second"] or float("nan")
    decode_ips = OPENFAKE_SAMPLE["decode_images_per_second"] or float("nan")
    out = {"shards_in_pool": len(ordered), "mean_shard_gb": round(mean_shard_gb, 2), "mean_rows_per_shard": round(mean_rows, 1),
           "shards_are_shuffled_note": "whole shards are read in a fixed seeded order; a quota for a generator is filled "
                                       "by reading shards until enough of its rows have gone by",
           "families_total_fakes_in_pool": dict(total_family.most_common()), "quotas": {}}
    for quota in (250, 500, 1000, 2000, 4000):
        need = shards_to_fill_quota(per_shard_family, quota)
        reachable = {f: n for f, n in need.items() if n is not None}
        out["quotas"][str(quota)] = {
            "families_that_reach_it": len(reachable), "families_that_never_do": sorted(f for f, n in need.items() if n is None),
            "shards_needed_per_family": reachable,
            "shards_for_the_rarest_reachable_family": max(reachable.values()) if reachable else None,
        }
    # Hours for reading K whole shards: one pass downloads every row group; decode time assumes every row is decoded
    # (an upper bound -- thinning the biggest generators lowers it). Pass 2 (CLIP) downloads the same bytes again.
    plans = []
    for k in sorted({10, 20, 40, 80, 160, 320, len(ordered)}):
        if k > len(ordered):
            continue
        gb, rows = k * mean_shard_gb, k * mean_rows
        download_h, decode_h = gb * 1000 / mbps / 3600, rows / decode_ips / 3600
        timed = bool(np.isfinite(download_h + decode_h))   # False only if the sample above measured no time at all
        families_min = min((sum((c for c in per_shard_family[:k]), Counter())[f] for f in total_family), default=0)
        plans.append({"shards": k, "gb_per_pass": round(gb, 0), "rows": int(rows),
                      "download_hours_per_pass": round(download_h, 2) if timed else None,
                      "decode_hours_if_every_row_decoded": round(decode_h, 2) if timed else None,
                      "hours_per_pass_sequential": round(download_h + decode_h, 2) if timed else None,
                      "fits_one_session_per_pass": bool(download_h + decode_h < TIME_BUDGET_HOURS) if timed else None,
                      "sessions_needed_for_both_passes": int(np.ceil(2 * (download_h + decode_h) / TIME_BUDGET_HOURS)) if timed else None,
                      "fakes_of_the_rarest_family": int(families_min)})
    out["plans"] = plans
    out["pass_rule"] = ("the planned sample must fit in sessions that resume from chunked writes, none longer than "
                        f"{TIME_BUDGET_HOURS} h (Kaggle's limit is 12 h); this table is the evidence, the choice is made after reading it")
    print(json.dumps({k: v for k, v in out.items() if k != "quotas"}, indent=1))
    for quota, q in out["quotas"].items():
        print(f"quota {quota:>5s} fakes per family: {q['families_that_reach_it']} families reach it; rarest of those needs "
              f"{q['shards_for_the_rarest_reachable_family']} shards; never reached by: {q['families_that_never_do']}")
    return out


preflight_step("quota_feasibility", check_quota_feasibility)

In [ ]:
# --- 0f. ArtiFact and FFHQ: what is attached, and what it really contains -----------------------------
# Found by folder name under /kaggle/input. If a name does not match (the environment check above prints the
# attached folders), write the folder's path into the override instead.
ARTIFACT_DIR_OVERRIDE = None
FFHQ_DIR_OVERRIDE = None
_inputs = find_kaggle_inputs()
ARTIFACT_DIR = ARTIFACT_DIR_OVERRIDE or os.environ.get("TRUSTMESH_ARTIFACT_DIR") or match_input(_inputs, ["artifact"])
FFHQ_DIR = FFHQ_DIR_OVERRIDE or os.environ.get("TRUSTMESH_FFHQ_DIR") or match_input(_inputs, ["ffhq"], ARTIFACT_DIR) \
    or match_input(_inputs, ["flickr", "faces"], ARTIFACT_DIR) or match_input(_inputs, ["thumbnails"], ARTIFACT_DIR)


def check_artifact():
    if not ARTIFACT_DIR:
        return {"attached": False, "note": "no attached dataset folder has 'artifact' in its name -- add awsaf49/artifact-dataset as an input"}
    summary = summarize_folder(ARTIFACT_DIR, max_seconds=60 if SMOKE_TEST else 900)
    out = {"attached": True, "path": ARTIFACT_DIR, "files": summary["files"], "gb": round(summary["bytes"] / 1e9, 2),
           "walk_complete": summary["complete"], "walk_seconds": round(summary["seconds"], 1),
           "top_level_folders": len(summary["by_top_dir"]),
           "by_folder": {k: v for k, v in sorted(summary["by_top_dir"].items())}, "csv_files": summary["csv_files"][:80]}
    described = {}
    for rel in summary["csv_files"][:80]:
        try:
            described[rel] = describe_csv(os.path.join(ARTIFACT_DIR, rel))
        except Exception as exc:  # noqa: BLE001
            described[rel] = {"error": f"{type(exc).__name__}: {exc}"[:200]}
    out["csv"] = described
    try:
        index = index_artifact(ARTIFACT_DIR)
        frame = pd.DataFrame(index)
        out["index_rows"] = int(len(frame))
        out["real_images"] = int((frame["is_fake"] == 0).sum())
        out["fake_images"] = int((frame["is_fake"] == 1).sum())
        out["per_folder_class_counts"] = {str(g): {"real": int((p["is_fake"] == 0).sum()), "fake": int((p["is_fake"] == 1).sum())}
                                         for g, p in frame.groupby("generator")}
        out["fake_generator_folders"] = sorted(frame[frame["is_fake"] == 1]["generator"].unique().tolist())
        out["real_source_folders"] = sorted(frame[frame["is_fake"] == 0]["generator"].unique().tolist())
        sample = frame.groupby("generator", group_keys=False).head(60)["path"].tolist()
        out["header_sample"] = header_summary(sample, threads=THREADS * 2)
    except Exception as exc:  # noqa: BLE001 -- the README's layout did not hold; the raw listing above is the record
        out["index_error"] = f"{type(exc).__name__}: {exc}"[:400]
    print(json.dumps({k: v for k, v in out.items() if k not in ("csv", "by_folder")}, indent=1))
    for rel, d in list(described.items())[:40]:
        print(f"  {rel}: columns {d.get('columns')} rows {d.get('rows')} target {d.get('value_counts', {}).get('target')}")
    return out


def check_ffhq():
    if not FFHQ_DIR:
        return {"attached": False, "note": "no attached dataset folder matches 'ffhq' / 'flickr faces' -- add the FFHQ thumbnails dataset as an input"}
    paths = list_images(FFHQ_DIR)
    out = {"attached": True, "path": FFHQ_DIR, "image_files": len(paths),
           "gb": round(sum(os.path.getsize(p) for p in paths) / 1e9, 2),
           "headers": header_summary(paths, threads=THREADS * 2),
           "license_on_kaggle_page": "Unknown (as seen by the user, 2026-10-06)",
           "license_at_source": "NVlabs/ffhq-dataset README: CC BY-NC-SA 4.0 (as cited in the spec; not re-read here)"}
    out["expected_70000_files"] = out["image_files"] == 70000
    out["all_128x128_rgb"] = out["headers"]["sizes"] == {"128x128": len(paths)} and set(out["headers"]["modes"]) == {"RGB"}
    print(json.dumps(out, indent=1))
    return out


preflight_step("artifact", check_artifact)
preflight_step("ffhq", check_ffhq)

In [ ]:
# --- 0g. Source versus label: can "which dataset" or "what size" give the label away? ------------------
def check_source_vs_label():
    out = {}
    counts = PREFLIGHT.get("openfake_counts") or {}
    if counts:
        by = counts["rows_by_split_and_label"]
        out["openfake"] = {"real": sum(v for k, v in by.items() if k.endswith("/real")),
                           "fake": sum(v for k, v in by.items() if k.endswith("/fake")),
                           "sizes_in_streamed_sample": (PREFLIGHT.get("streaming_sample") or {}).get("sizes"),
                           "formats_in_streamed_sample": (PREFLIGHT.get("streaming_sample") or {}).get("format")}
    art = PREFLIGHT.get("artifact") or {}
    if art.get("attached"):
        out["artifact"] = {"real": art.get("real_images"), "fake": art.get("fake_images"),
                           "sizes_in_header_sample": (art.get("header_sample") or {}).get("sizes"),
                           "formats_in_header_sample": (art.get("header_sample") or {}).get("formats")}
    ffhq = PREFLIGHT.get("ffhq") or {}
    if ffhq.get("attached"):
        out["ffhq"] = {"real": ffhq.get("image_files"), "fake": 0, "sizes": ffhq["headers"]["sizes"],
                       "formats": ffhq["headers"]["formats"]}
    out["one_class_sources"] = [k for k, v in out.items() if isinstance(v, dict) and (not v.get("real") or not v.get("fake"))]
    out["note"] = ("a source that contributes only one class makes 'source' (and its image size) a label shortcut; "
                   "Task 4's source-only baseline measures it and decides what happens to that source")
    print(json.dumps(out, indent=1))
    return out


preflight_step("source_vs_label", check_source_vs_label)

In [ ]:
# --- 0h. CLIP ViT-L/14: is it loadable, what does it output, how fast is it ---------------------------
CLIP_ENCODER = None


def check_clip():
    global CLIP_ENCODER
    from transformers import CLIPImageProcessor

    t0 = time.time()
    if SMOKE_TEST:
        CLIP_ENCODER = ClipEncoder(model=tiny_clip_for_smoke(), device=DEVICE, fp16=False)
        processor = CLIPImageProcessor()
    else:
        CLIP_ENCODER = ClipEncoder(CLIP_MODEL, device=DEVICE, fp16=True)
        processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL)
    config = CLIP_ENCODER.model.config
    out = {"model": "tiny random CLIP (smoke test)" if SMOKE_TEST else CLIP_MODEL, "loader": "transformers.CLIPVisionModelWithProjection",
           "load_seconds": round(time.time() - t0, 1), "image_size": int(config.image_size), "patch_size": int(config.patch_size),
           "hidden_size_pre_projection": int(config.hidden_size), "projection_dim": int(config.projection_dim),
           "official_preprocessing": {k: str(getattr(processor, k, None)) for k in
                                      ("size", "crop_size", "resample", "do_center_crop", "do_normalize", "image_mean", "image_std",
                                       "do_convert_rgb", "rescale_factor")}}
    # The embedding this engine stores, measured rather than read from the config.
    picture = synthetic_picture(np.random.default_rng(0), False, (640, 480))
    pixels = clip_pixels(resize_center_crop(picture)).unsqueeze(0)
    vector = CLIP_ENCODER.embed(pixels)
    out["embedding_shape_produced"] = list(vector.shape)
    out["embedding_is"] = "the projected image embedding (image_embeds), not L2-normalized, stored as float32"
    with torch.no_grad():
        raw = CLIP_ENCODER.model(pixel_values=pixels.to(DEVICE).half() if CLIP_ENCODER.half else pixels.to(DEVICE))
    out["pre_projection_pooled_dim"] = int(raw.last_hidden_state.shape[-1])
    # Our preprocessing (no JPEG step) against the official processor on the same picture.
    official = processor(images=picture, return_tensors="pt")["pixel_values"][0]
    out["official_processor_class"] = type(processor).__name__
    out["max_abs_difference_vs_official_preprocessing"] = float((official - pixels[0]).abs().max())
    out["one_grey_level_in_normalized_units"] = round(1 / 255 / min(CLIP_STD), 4)
    try:   # the PIL-backed official processor resamples exactly as this code does; the default one may use torchvision
        from transformers import CLIPImageProcessorPil

        official_pil = CLIPImageProcessorPil()(images=picture, return_tensors="pt")["pixel_values"][0]
        out["max_abs_difference_vs_official_pil_preprocessing"] = float((official_pil - pixels[0]).abs().max())
    except Exception as exc:  # noqa: BLE001 -- older transformers have no separate PIL class
        out["max_abs_difference_vs_official_pil_preprocessing"] = f"not available: {type(exc).__name__}"
    batch = torch.stack([preprocess_image(synthetic_picture(np.random.default_rng(i), i % 2 == 1)) for i in range(4)])
    out["determinism"] = embedding_is_deterministic(CLIP_ENCODER, batch)
    n_batches, size = (2, 8) if SMOKE_TEST else (6, EMBED_BATCH)
    speed_batch = torch.randn(size, 3, 224, 224)
    CLIP_ENCODER.embed(speed_batch)
    t0 = time.perf_counter()
    for _ in range(n_batches):
        CLIP_ENCODER.embed(speed_batch)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    out["encoder_images_per_second"] = round(n_batches * size / max(time.perf_counter() - t0, 1e-9), 1)
    out["encoder_fp16"] = CLIP_ENCODER.half
    print(json.dumps(out, indent=1))
    return out


preflight_step("clip", check_clip)

In [ ]:
# --- 0i. Disk budget for the embedding cache; Deepfake-Eval-2024 access; Synthbuster ------------------
def check_disk_budget():
    measured = (PREFLIGHT.get("clip") or {}).get("embedding_shape_produced")
    dim = measured[-1] if measured else 768
    f = PLAN["partition_fractions"]
    vectors_per_image = f["train"] * (1 + N_AUG_VIEWS) + f["val"] + (f["known_test"] + f["holdout"]) * (1 + len(DEGRADATIONS))
    out = {"embedding_dim_used": int(dim),
           "embedding_dim_source": "measured in the CLIP check above" if measured else
                                   "ASSUMED 768 from the handbook, because the CLIP check above did not produce one",
           "bytes_per_vector_float32": int(dim) * 4,
           "vectors_per_image_average": round(vectors_per_image, 2),
           "how": f"train images: 1 clean + {N_AUG_VIEWS} augmented views; validation: 1; test images: 1 clean + "
                  f"{len(DEGRADATIONS)} degradations ({', '.join(DEGRADATIONS)})",
           "projected_cache_gb": {str(n): round(n * vectors_per_image * dim * 4 / 1e9, 2) for n in (60000, 100000, 150000, 250000, 500000)},
           "metadata_table_size": "not yet measured (the scan stage reports it)",
           "disk_now": disk_report(), "images_written_to_disk": 0}
    print(json.dumps(out, indent=1))
    return out


def check_deepfake_eval_access():
    if SMOKE_TEST:
        return {"status": "not checked (smoke test)"}
    from huggingface_hub import HfApi

    api = HfApi(token=_token)
    out = {"repo": DEEPFAKE_EVAL_REPO, "user_submitted_request": "2026-10-06 (per the spec)", "use": "evaluation only; never training or tuning"}
    try:
        if hasattr(api, "auth_check"):
            api.auth_check(DEEPFAKE_EVAL_REPO, repo_type="dataset")
        else:
            api.list_repo_files(DEEPFAKE_EVAL_REPO, repo_type="dataset")
        out["status"] = "access granted to this token"
    except Exception as exc:  # noqa: BLE001
        name = type(exc).__name__
        out["status"] = ("gated: access not granted to this token yet" if "Gated" in name else
                         "repository not found under this id (check the id)" if "NotFound" in name else f"check failed: {name}")
    print(json.dumps(out, indent=1))
    return out


preflight_step("disk_budget", check_disk_budget)
preflight_step("deepfake_eval_2024", check_deepfake_eval_access)
save_preflight("synthbuster", {
    "status": "not attached -- not needed until the end of V1 (the user uploads it to a private Kaggle dataset later)",
    "what_it_is": "Zenodo record 10066460: 9,000 fake images from nine generators, no real images, CC-BY-NC-SA 4.0 (per the spec)",
    "consequence_of_no_reals": "on its own it supports only the detection rate on fakes. ROC-AUC, EER or FPR need real images "
                               "from another source: the reserved real pool of this notebook (never trained on) is the planned "
                               "source, and any such number carries the caveat that reals and fakes come from different sources."})
save_preflight("run_hours", round(elapsed_hours(), 2))
print("\n" + "=" * 100)
print(f"Task 0 finished in {elapsed_hours():.2f} h. Failed checks: {list(PREFLIGHT['failed_steps']) or 'none'}")
print(f"Wrote {PREFLIGHT_PATH}, openfake_model_counts.csv and openfake_shard_counts.csv (aggregate counts only).")
if STAGE == "task0":
    print("STAGE is 'task0': nothing below this cell runs. Report this log and preflight_task0.json back before any extraction.")
print("=" * 100)

## Stage control for everything after Task 0

A stage runs only if `STAGE` reaches it, nothing earlier has stopped the run, and every `PLAN`
value it needs has been decided. A stage that cannot run says why and the notebook finishes
cleanly -- it never raises, because a failed Kaggle version keeps none of its output.

In [ ]:
STOPPED = None
SHARD_SIZES = {s["path"]: s["size"] for s in OPENFAKE_SHARDS}


def halt(reason):
    global STOPPED
    if STOPPED is None:
        STOPPED = reason
        RESULTS["not_run"]["halted"] = reason
        save_results("not_run", RESULTS["not_run"])
    print("HALTED:", reason)


def ready(stage, *keys):
    if not stage_on(stage):
        print(f"stage '{stage}': not yet run (STAGE is '{STAGE}')")
        return False
    if STOPPED:
        print(f"stage '{stage}': not run -- {STOPPED}")
        return False
    missing = [k for k in keys if PLAN.get(k) is None]
    if missing:
        halt(f"stage '{stage}' needs PLAN{missing}, which are decided from the Task 0 results and are still None")
        return False
    return True


def chunk_files(subdir, pattern):
    """Finished chunk files of a stage: this session's, then those of attached earlier sessions."""
    found = {}
    for base in [PRIVATE_DIR] + [Path(d) for d in RESUME_DIRS]:
        for path in sorted((base / subdir).glob(pattern)):
            found.setdefault(path.name, path)
    return found


save_results("plan", {k: (list(v) if isinstance(v, tuple) else v) for k, v in PLAN.items()})

## Tasks 1-2 (stage `scan`) -- pass 1: one metadata row per image

Each selected image is read once: validity, file properties as received, perceptual hash, and the
byte count after the shared pipeline. No encoder runs. One chunk file per shard (or per block of
files), so a session that stops is resumed by attaching its output as an input.

In [ ]:
SCAN_COMPLETE = False
if ready("scan", "openfake_shards", "keep_fraction", "artifact_per_folder", "ffhq_images"):
    scan_dir = PRIVATE_DIR / "scan"
    scan_dir.mkdir(parents=True, exist_ok=True)
    reusable = chunk_files("scan", "rows-*.parquet")
    pool = [s for s in OPENFAKE_SHARDS if s["config"] == "core" and s["split"] in PLAN["openfake_splits"]]
    work = []
    for shard in seeded_shard_order(pool)[:PLAN["openfake_shards"]]:
        key = "openfake-" + shard["path"].rsplit("/", 1)[-1].replace(".parquet", "")
        work.append((key, lambda shard=shard: scan_openfake_shard(OPENFAKE_READER, shard, PLAN["keep_fraction"], SEED, THREADS)))
    folder_index = []
    if ARTIFACT_DIR and PLAN["artifact_per_folder"]:
        by_folder = {}
        for row in index_artifact(ARTIFACT_DIR):
            by_folder.setdefault(row["generator"], []).append(row)
        for rows in by_folder.values():
            folder_index += sorted(rows, key=lambda r: stable_fraction(r["image_id"], SEED))[:PLAN["artifact_per_folder"]]
    if FFHQ_DIR and PLAN["ffhq_images"]:
        folder_index += sorted(index_ffhq(FFHQ_DIR), key=lambda r: stable_fraction(r["image_id"], SEED))[:PLAN["ffhq_images"]]
    block = 20 if SMOKE_TEST else 5000
    for i in range(0, len(folder_index), block):
        part = folder_index[i:i + block]
        work.append((f"folders-{i // block:05d}", lambda part=part: (scan_folder_rows(part, THREADS), {"kept": len(part)})))

    scan_log = {"chunks_total": len(work), "chunks_reused": 0, "chunks_written": 0, "rows_written": 0}
    SCAN_COMPLETE = True
    for n, (key, fn) in enumerate(work, 1):
        name = f"rows-{key}.parquet"
        if name in reusable:
            scan_log["chunks_reused"] += 1
            continue
        if not within_budget("scan"):
            SCAN_COMPLETE = False
            break
        t0 = time.time()
        rows, stats = fn()
        build_table(rows).to_parquet(scan_dir / (name + ".tmp"), index=False)
        os.replace(scan_dir / (name + ".tmp"), scan_dir / name)
        scan_log["chunks_written"] += 1
        scan_log["rows_written"] += len(rows)
        print(f"scan {n}/{len(work)} {key}: {len(rows)} rows {stats} in {time.time() - t0:.0f}s | elapsed {elapsed_hours():.2f} h")
    scan_log["complete"] = SCAN_COMPLETE
    save_results("scan", scan_log)
    print("scan:", scan_log)
    if not SCAN_COMPLETE:
        halt("the scan stopped at the time budget; attach this version's output as an input and run again to resume")

## Task 2 (stage `split`) -- cleaning, and the evidence for the dedupe tolerance

In [ ]:
TABLE, FOLDS, OLD_FAMILIES, SHORTCUT, GATE_OK = None, [], [], None, False
if ready("split"):
    files = chunk_files("scan", "rows-*.parquet")
    if not files:
        halt("no scan chunks found: run the scan stage first (or attach its output as an input)")
    else:
        TABLE = pd.concat([pd.read_parquet(p) for p in files.values()], ignore_index=True).drop_duplicates("image_id")
        TABLE = TABLE.reset_index(drop=True)
        TABLE["valid"] = TABLE["valid"].astype(bool)
        histogram = distance_histogram(TABLE)
        cleaning = {"scan_chunks": len(files), "table_mb_in_memory": round(TABLE.memory_usage(deep=True).sum() / 1e6, 1),
                    "cleaning": cleaning_report(TABLE), "null_counts": null_counts(TABLE),
                    "sources_before_dedupe": source_class_table(TABLE),
                    "pairs_at_each_hash_distance": histogram}
        save_results("cleaning", cleaning)
        print(json.dumps(cleaning, indent=1))
        if SHOW_DUPLICATE_PAIRS:
            # Private notebooks only: this draws dataset images into the notebook's rendered output.
            import io as _io

            import matplotlib.pyplot as plt
            from PIL import Image as _Image

            examples = duplicate_examples(TABLE, 7, per_distance=2)[:10]
            by_id = TABLE.set_index("image_id")
            wanted = sorted({i for a, b, _ in examples for i in (a, b)})
            blobs = fetch_image_bytes([(i, by_id.loc[i, "source_dataset"], by_id.loc[i, "locator"]) for i in wanted],
                                      OPENFAKE_READER, SHARD_SIZES)
            if examples:
                fig, axes = plt.subplots(len(examples), 2, figsize=(5, 2.4 * len(examples)), squeeze=False)
                for axis_row, (a, b, d) in zip(axes, examples):
                    for ax, image_id in zip(axis_row, (a, b)):
                        ax.imshow(_Image.open(_io.BytesIO(blobs[image_id])).convert("RGB"))
                        ax.set_title(f"distance {d}", fontsize=7)
                        ax.axis("off")
                plt.tight_layout()
                plt.show()
        if PLAN["phash_tolerance"] is None:
            halt("PLAN['phash_tolerance'] is not chosen yet: pick it from the pair counts above (and a look at real pairs)")

## Tasks 3-4 (stage `split`) -- dedupe, rotating held-out families, leakage checks, shortcut baselines

Groups (shared prompt, source image or identity) go whole into one partition. Each eligible family
gets a fold in which all its fakes are unseen. The shortcut baselines then ask whether file
properties or the source alone give the label away; the embedding stage does not start unless
they cannot, or the residual has been read and accepted explicitly.

In [ ]:
if TABLE is not None and ready("split", "phash_tolerance", "caps", "min_unseen_fakes", "old_families"):
    TABLE, dedupe_report = dedupe_table(TABLE, PLAN["phash_tolerance"])
    TABLE = assign_partitions(TABLE, PLAN["partition_fractions"], SEED)
    TABLE = apply_generator_caps(TABLE, PLAN["caps"], SEED)
    eligibility = eligible_families(TABLE, PLAN["min_unseen_fakes"])
    FOLDS = [f for f, e in eligibility.items() if e["eligible"]]
    present = set(TABLE.loc[TABLE["valid"], "generator_family"])
    OLD_FAMILIES = [f for f in PLAN["old_families"] if f in present]
    TABLE = add_fold_roles(TABLE, FOLDS, SEED, old_families=OLD_FAMILIES or None)
    leakage = leakage_report(TABLE, PLAN["phash_tolerance"], strict=False)
    live = TABLE[TABLE["valid"]]
    split_report = {
        "dedupe": dedupe_report, "rows_valid_after_dedupe_and_caps": int(len(live)),
        "dropped_by_reason": {str(k): int(v) for k, v in TABLE.loc[~TABLE["valid"], "drop_reason"].value_counts().items()},
        "partition_counts": {f"{s}/{'fake' if f else 'real'}": int(n) for (s, f), n in live.groupby(["split", "is_fake"]).size().items()},
        "groups": int(live["group_id"].nunique()), "family_eligibility": eligibility, "folds": FOLDS,
        "old_families_for_ablation": OLD_FAMILIES, "reserved_real_pool": int(((live["is_fake"] == 0) & (live["split"] == "holdout")).sum()),
        "leakage_violations": leakage["violations"],
        "leakage_per_fold": {f: {k: v for k, v in e.items() if k != "balance"} for f, e in leakage["folds"].items()},
        "class_balance_per_fold": {f: e["balance"]["per_role"] for f, e in leakage["folds"].items()},
        "generator_counts_per_fold": {f: e["balance"]["per_generator"] for f, e in leakage["folds"].items()},
        "sources_after_dedupe": source_class_table(TABLE),
        "auc_ci_halfwidth_at_fold_sizes": {
            f: {str(auc): round(auc_ci_halfwidth(auc, e["holdout"], min(e["holdout"], e["reserved_reals"])), 4) for auc in (0.7, 0.9, 0.99)}
            for f, e in eligibility.items() if e["eligible"]},
    }
    print(json.dumps({k: v for k, v in split_report.items() if k not in ("generator_counts_per_fold", "leakage_per_fold")}, indent=1))
    for fold, balance in split_report["class_balance_per_fold"].items():
        print(f"fold {fold:24s} " + " | ".join(f"{role}: {c.get('real', 0)} real / {c.get('fake', 0)} fake" for role, c in balance.items()))
    if leakage["violations"]:
        halt("leakage checks failed: " + "; ".join(leakage["violations"]))
    elif not FOLDS:
        halt("no family is eligible to be held out at PLAN['min_unseen_fakes']")

    # Task 4: shortcut baselines, before any encoder runs.
    SHORTCUT = run_shortcut_baselines(TABLE, N_BOOTSTRAP, SEED)
    print(f"\n{'features':10s} {'model':8s} {'ROC-AUC':>8s}  95% CI            separable   train/test")
    for r in SHORTCUT["results"]:
        if r["auc"] is None:
            print(f"{r['mode']:10s} {r['kind']:8s} not scored: {r.get('note')}")
        else:
            print(f"{r['mode']:10s} {r['kind']:8s} {r['auc']:8.4f}  [{r['ci_low']:.4f}, {r['ci_high']:.4f}]  {str(r['separable']):9s}  "
                  f"{r['n_train']}/{r['n_test']}")
    GATE_OK = bool(SHORTCUT["gate"]["passed"] or ACCEPT_RESIDUAL_SHORTCUT)
    split_report["shortcut_baselines"] = SHORTCUT
    split_report["shortcut_gate"] = {**SHORTCUT["gate"], "residual_accepted_explicitly": bool(ACCEPT_RESIDUAL_SHORTCUT and not SHORTCUT["gate"]["passed"])}
    print("shortcut gate:", json.dumps(split_report["shortcut_gate"], indent=1))
    TABLE.to_parquet(PRIVATE_DIR / "metadata.parquet", index=False)   # per-image rows: private output only
    split_report["metadata_table_mb_on_disk"] = round((PRIVATE_DIR / "metadata.parquet").stat().st_size / 1e6, 2)
    save_results("split", split_report)

## Task 5 (stage `extract`) -- frozen CLIP embeddings

Every image goes through the same pipeline (decode -> RGB -> 224x224 -> one JPEG re-encode -> CLIP
normalization). Training images get the clean embedding plus augmented views; validation images
only the clean one; test images the clean one plus each fixed degradation. Only vectors are saved.

In [ ]:
EXTRACT_COMPLETE, STORE = False, None
if TABLE is not None and ready("extract"):
    if not GATE_OK:
        halt("the shortcut baselines separate the classes after the shared pipeline. Fix the pipeline (or rebalance sources) "
             "and re-run them; or, having read the numbers, set ACCEPT_RESIDUAL_SHORTCUT = True and carry the caveat")
    else:
        if CLIP_ENCODER is None:
            CLIP_ENCODER = ClipEncoder(model=tiny_clip_for_smoke(), device=DEVICE, fp16=False) if SMOKE_TEST else \
                ClipEncoder(CLIP_MODEL, device=DEVICE, fp16=True)
        VARIANTS = {"train": ["clean"] + [f"aug{k}" for k in range(N_AUG_VIEWS)], "val": ["clean"],
                    "known_test": ["clean"] + list(DEGRADATIONS), "holdout": ["clean"] + list(DEGRADATIONS)}
        live = TABLE[TABLE["valid"]]
        chunks, group_block, file_block = [], 4 if SMOKE_TEST else 8, 40 if SMOKE_TEST else 2000
        openfake = live[live["source_dataset"] == "openfake"]
        by_block = {}
        for image_id, locator, split in zip(openfake["image_id"], openfake["locator"], openfake["split"]):
            path, group, _ = locator.rsplit("#", 2)
            by_block.setdefault((path, int(group) // group_block), {})[locator] = (image_id, VARIANTS[split])
        for (path, block_index), wanted in sorted(by_block.items()):
            key = f"openfake-{path.rsplit('/', 1)[-1].replace('.parquet', '')}-g{block_index:03d}"
            chunks.append((key, lambda path=path, wanted=wanted: load_openfake_items(
                OPENFAKE_READER, {"path": path, "size": SHARD_SIZES[path]}, wanted)))
        files = live[live["source_dataset"] != "openfake"].sort_values("image_id")
        file_rows = [(i, p, VARIANTS[s]) for i, p, s in zip(files["image_id"], files["locator"], files["split"])]
        for i in range(0, len(file_rows), file_block):
            chunks.append((f"folders-{i // file_block:05d}", lambda part=file_rows[i:i + file_block]: load_folder_items(part)))

        STORE = ChunkStore(PRIVATE_DIR / "embeddings", "emb", [Path(d) / "embeddings" for d in RESUME_DIRS])
        probe = torch.stack([preprocess_image(synthetic_picture(np.random.default_rng(i), i % 2 == 1)) for i in range(4)])
        determinism = embedding_is_deterministic(CLIP_ENCODER, probe)

        def chunks_within_budget():
            for key, load in chunks:
                if not STORE.has(key) and not within_budget("extract"):
                    return
                yield key, load

        extract_report = extract_embeddings(CLIP_ENCODER, chunks_within_budget(), STORE, EMBED_BATCH, SEED, THREADS)
        EXTRACT_COMPLETE = all(STORE.has(key) for key, _ in chunks)
        extract_report.update(chunks_total=len(chunks), complete=EXTRACT_COMPLETE, determinism_check=determinism,
                              encoder=CLIP_ENCODER.name, embedding_dim=CLIP_ENCODER.dim, variants=VARIANTS,
                              images_to_embed=int(len(live)), images_written_to_disk=0, disk_after=disk_report())
        save_results("extract", extract_report)
        print(json.dumps({k: v for k, v in extract_report.items() if k != "variants"}, indent=1))
        if not EXTRACT_COMPLETE:
            halt("embedding extraction stopped at the time budget; attach this version's output as an input and run again to resume")

## Tasks 6-8 (stage `heads`) -- heads, calibration, evaluation suites

Per fold: logistic regression (C chosen on that fold's validation split) and a one-hidden-layer
MLP. Temperature, operating threshold and uncertain band are fitted on the fold's validation
split, which holds seen generators only. Fake is the positive class. Confidence intervals are 95%
stratified bootstraps on the first seed; further seeds are run only where a seed changes something.

In [ ]:
FOLD_RESULTS, FOLD_EXTRAS, BANK = {}, {}, None
HEAD_KWARGS = dict(n_aug=N_AUG_VIEWS, n_resamples=N_BOOTSTRAP, target_recall=TARGET_RECALL, band_recall=BAND_RECALL,
                   band_fpr=BAND_FPR, device=DEVICE)


def run_seeds(fold, kind, extra_tests=None):
    """First seed with CIs; more seeds (point values only) unless the fit is deterministic."""
    runs, extras = [], None
    for k, seed in enumerate(SEEDS):
        result, extra = run_fold(TABLE, BANK, fold, kind, seed, with_ci=(k == 0), extra_tests=extra_tests, **HEAD_KWARGS)
        runs.append(result)
        if k == 0:
            extras = extra
        if result.get("not_run") or result["seed_changes"].startswith("nothing"):
            break
    suites = sorted({s for r in runs for s in r.get("suites", {})})
    across = {s: {m: mean_std([metric_value(r, s, m) for r in runs]) for m in ("roc_auc", "eer")} for s in suites}
    return {"first_seed": runs[0], "seeds_run": [r["seed"] for r in runs], "seed_changes": runs[0]["seed_changes"],
            "across_seeds": across}, extras


def cell(result, suite, metric="roc_auc"):
    """'0.9731 [0.9690, 0.9770]' for a results table, or 'not yet run'."""
    try:
        m = result["first_seed"]["suites"][suite]["metrics"][metric]
        return f"{m['value']:.4f} [{m['ci_low']:.4f}, {m['ci_high']:.4f}]"
    except (KeyError, TypeError):
        return NOT_RUN


if TABLE is not None and ready("heads"):
    if STORE is None or not EXTRACT_COMPLETE:
        halt("the embedding cache is not complete; finish the extract stage first")
    else:
        BANK = EmbeddingBank(*STORE.load())
        print(f"embedding bank: {len(BANK.index)} vectors of {BANK.dim} numbers; variants {BANK.variants()}")
        for fold in FOLDS:
            FOLD_RESULTS[fold], FOLD_EXTRAS[fold] = {}, {}
            for kind in ("logreg", "mlp"):
                t0 = time.time()
                FOLD_RESULTS[fold][kind], FOLD_EXTRAS[fold][kind] = run_seeds(fold, kind)
                print(f"fold {fold:20s} {kind:6s} known AUC {cell(FOLD_RESULTS[fold][kind], 'known')} | unseen AUC "
                      f"{cell(FOLD_RESULTS[fold][kind], 'unseen')} | seeds {FOLD_RESULTS[fold][kind]['seeds_run']} | {time.time() - t0:.0f}s")
            # MLP against logistic regression on the same test images (first seeds).
            versus = {}
            for suite in ("known", "unseen"):
                a = FOLD_EXTRAS[fold]["mlp"]["frames"].get(suite)
                b = FOLD_EXTRAS[fold]["logreg"]["frames"].get(suite)
                if a is not None and b is not None and len(a) == len(b):
                    versus[suite] = paired_metric_difference(a["prob"].to_numpy(), b["prob"].to_numpy(), a["is_fake"].to_numpy(),
                                                             "roc_auc", 0.5, N_BOOTSTRAP, SEED)
            FOLD_RESULTS[fold]["mlp_minus_logreg"] = versus
            save_results("folds", FOLD_RESULTS)

In [ ]:
# Old-only ablation (its own row, labelled an ablation): train on real images plus the old families
# only, then score it on the unseen-test images of every other fold -- the same images those folds use.
OLD_ONLY_RESULT = None
if BANK is not None and ready("heads") and FOLD_PREFIX + OLD_ONLY in TABLE.columns:
    modern = [f for f in FOLDS if f not in OLD_FAMILIES]
    tests = {f"unseen[{f}]": fold_rows(TABLE, f, "unseen_test") for f in modern}
    OLD_ONLY_RESULT, old_extras = run_seeds(OLD_ONLY, "logreg", extra_tests=tests)
    versus = {}
    for f in modern:
        a = old_extras["frames"].get(f"unseen[{f}]")
        b = FOLD_EXTRAS[f]["logreg"]["frames"].get("unseen")
        if a is not None and b is not None:
            merged = a.merge(b[["image_id", "prob"]], on="image_id", suffixes=("_old_only", "_lofo"))
            versus[f] = paired_metric_difference(merged["prob_old_only"].to_numpy(), merged["prob_lofo"].to_numpy(),
                                                 merged["is_fake"].to_numpy(), "roc_auc", 0.5, N_BOOTSTRAP, SEED)
        print(f"old-only -> {f:20s} unseen AUC {cell(OLD_ONLY_RESULT, f'unseen[{f}]')} | LOFO fold on the same images "
              f"{cell(FOLD_RESULTS[f]['logreg'], 'unseen')}")
    OLD_ONLY_RESULT["old_only_minus_lofo_on_same_images"] = versus
    OLD_ONLY_RESULT["old_families"] = OLD_FAMILIES
    save_results("old_only_ablation", OLD_ONLY_RESULT)
elif stage_on("heads") and BANK is not None:
    RESULTS["not_run"]["old_only_ablation"] = "none of PLAN['old_families'] is present in the data"
    save_results("not_run", RESULTS["not_run"])

In [ ]:
# Final model: the head retrained on every family, including those held out in the folds. It has no
# untouched unseen test of its own -- the folds above are its generalization *estimate*, not a measurement.
FINAL_RESULT = None
if BANK is not None and ready("heads"):
    final_fold = "final_all_families"
    TABLE[FOLD_PREFIX + final_fold] = [s if ok and s in ("train", "val", "known_test") else ("unused" if ok else None)
                                       for s, ok in zip(TABLE["split"], TABLE["valid"])]
    FINAL_RESULT, final_extras = run_seeds(final_fold, "logreg")
    weights_dir = PRIVATE_DIR / "weights"
    weights_dir.mkdir(exist_ok=True)
    hashes = {}
    if final_extras["head"] is not None:
        hashes["v1_final_logreg.npz"] = save_head(weights_dir / "v1_final_logreg.npz", final_extras["head"])
    for fold in FOLDS:
        for kind in ("logreg", "mlp"):
            head = FOLD_EXTRAS[fold][kind]["head"]
            if head is not None:
                name = f"v1_fold_{fold}_{kind}.npz".replace("/", "_")
                hashes[name] = save_head(weights_dir / name, head)
    FINAL_RESULT["weights_sha256"] = hashes
    FINAL_RESULT["weights_location"] = "artifacts/visual/private/weights (kept out of any public repository until the dataset licenses are checked)"
    FINAL_RESULT["what_it_cannot_claim"] = (
        "This head has seen every family, so it has no unseen-generator test of its own. Its known-test number is in-domain. "
        "The leave-one-family-out folds are an estimate of how it behaves on a new family, not a measurement of this model.")
    TABLE.drop(columns=[FOLD_PREFIX + final_fold], inplace=True)
    save_results("final_model", FINAL_RESULT)
    print(f"final model (all families): known AUC {cell(FINAL_RESULT, 'known')}; weights hashed: {len(hashes)} files")

## Task 9 -- heavier pixel baselines (off by default)

A small CNN from scratch and a fine-tuned ResNet-50, on a capped subset of images held in memory.
They run only when their flag is on, which is only after Tasks 1-8 have real results.

In [ ]:
PIXEL_RESULTS = {}
if BANK is not None and ready("heads") and (RUN_SMALL_CNN or RUN_RESNET50) and FOLDS:
    pixel_fold = FOLDS[0]
    pixel_caps = {"train": 24, "val": 12, "known_test": 12, "unseen_test": 12} if SMOKE_TEST else \
        {"train": 6000, "val": 1000, "known_test": 1500, "unseen_test": 1500}
    side = 64 if SMOKE_TEST else 224
    arrays = {}
    for role, cap in pixel_caps.items():
        rows = fold_rows(TABLE, pixel_fold, role)
        rows = rows.iloc[np.argsort([stable_fraction(i, SEED) for i in rows["image_id"]])]
        rows = pd.concat([rows[rows["is_fake"] == c].head(cap // 2) for c in (0, 1)])   # balanced, capped
        blobs = fetch_image_bytes(zip(rows["image_id"], rows["source_dataset"], rows["locator"]), OPENFAKE_READER, SHARD_SIZES)
        pixels = [(pixel_array(blobs[i], side), y) for i, y in zip(rows["image_id"], rows["is_fake"]) if i in blobs]
        pixels = [(x, y) for x, y in pixels if x is not None]
        arrays[role] = (np.stack([x for x, _ in pixels]), np.array([y for _, y in pixels], dtype=int))
    builders = {}
    if RUN_SMALL_CNN:
        builders["small_cnn"] = lambda: SmallCNN()
    if RUN_RESNET50:
        builders["resnet50"] = lambda: build_resnet50(pretrained=not SMOKE_TEST)
    for name, build in builders.items():
        print(f"training {name} on {len(arrays['train'][0])} images of fold {pixel_fold} ...")
        model, fit = train_pixel_model(build(), *arrays["train"], *arrays["val"], DEVICE, epochs=1 if SMOKE_TEST else 8, seed=SEED)
        entry = {"fold": pixel_fold, "fit": fit, "image_side": side, "images": {r: int(len(a[0])) for r, a in arrays.items()}, "suites": {}}
        for suite, role in (("known", "known_test"), ("unseen", "unseen_test")):
            x, y = arrays[role]
            probs = sigmoid(pixel_logits(model, x, DEVICE))
            entry["suites"][suite] = evaluate_suite(probs, y, 0.5, None, N_BOOTSTRAP, SEED)
        PIXEL_RESULTS[name] = entry
    save_results("pixel_baselines", PIXEL_RESULTS)
elif stage_on("heads"):
    RESULTS["not_run"]["pixel_baselines"] = "RUN_SMALL_CNN and RUN_RESNET50 are off (they run only after Tasks 1-8 have real results)"
    save_results("not_run", RESULTS["not_run"])

## Report: tables, plots, per-generator errors, closing paragraph

In [ ]:
SUITE_COLUMNS = ["known", "unseen", "known/jpeg90", "known/jpeg70", "known/jpeg50", "unseen/jpeg90", "unseen/jpeg70", "unseen/jpeg50"]
SUITE_NAMES = {"known": "T1 known generators", "unseen": "T3 unseen family", "known/jpeg90": "T4 known, JPEG 90",
               "known/jpeg70": "T4 known, JPEG 70", "known/jpeg50": "T4 known, JPEG 50", "unseen/jpeg90": "T4 unseen, JPEG 90",
               "unseen/jpeg70": "T4 unseen, JPEG 70", "unseen/jpeg50": "T4 unseen, JPEG 50"}
MODEL_ROWS = {"logreg": "V1 logistic regression", "mlp": "V1 MLP"}
SHORTCUT_ROW = "shortcut baseline (file properties after the shared pipeline)"

# The reporting grid: every cell is "not yet run" until a real run fills it.
report = {"metric": "ROC-AUC with 95% bootstrap CI (first seed)", "folds": {}, "deferred_suites": {
    "T2 sibling generator": "not run (run only if the real metadata has a clean sibling pair)",
    "T5 screenshots": NOT_RUN, "T6 resize": NOT_RUN + " (0.5x embeddings are cached by the extract stage)",
    "T7 crop": NOT_RUN + " (centre-crop embeddings are cached by the extract stage)", "T8 AI-edited pairs": NOT_RUN,
    "T9 real-world (Deepfake-Eval-2024)": NOT_RUN + " (once, after the V1 head is frozen, if access is granted)"}}
shortcut_cell = NOT_RUN
if SHORTCUT is not None:
    worst = max((r for r in SHORTCUT["results"] if r["mode"] == "pipeline" and r["auc"] is not None),
                key=lambda r: abs(r["auc"] - 0.5), default=None)
    if worst:
        shortcut_cell = f"{worst['auc']:.4f} [{worst['ci_low']:.4f}, {worst['ci_high']:.4f}] ({worst['kind']}, known-test partition, all families)"
for fold in (FOLDS or ["(folds not built yet)"]):
    grid = results_template([SHORTCUT_ROW] + list(MODEL_ROWS.values()), [SUITE_NAMES[s] for s in SUITE_COLUMNS])
    grid[SHORTCUT_ROW][SUITE_NAMES["known"]] = shortcut_cell
    for kind, row_name in MODEL_ROWS.items():
        for suite in SUITE_COLUMNS:
            if fold in FOLD_RESULTS and kind in FOLD_RESULTS[fold]:
                grid[row_name][SUITE_NAMES[suite]] = cell(FOLD_RESULTS[fold][kind], suite)
    for name, entry in PIXEL_RESULTS.items():
        if entry["fold"] == fold:
            grid[f"{name} (pixel baseline, capped subset)"] = {
                SUITE_NAMES[s]: (f"{entry['suites'][s]['metrics']['roc_auc']['value']:.4f} [{entry['suites'][s]['metrics']['roc_auc']['ci_low']:.4f}, "
                                 f"{entry['suites'][s]['metrics']['roc_auc']['ci_high']:.4f}]" if s in entry["suites"] else NOT_RUN)
                for s in SUITE_COLUMNS}
    report["folds"][fold] = grid
report["old_only_ablation"] = ({f: cell(OLD_ONLY_RESULT, f"unseen[{f}]") for f in FOLDS if f not in OLD_FAMILIES}
                               if OLD_ONLY_RESULT else NOT_RUN)
save_results("report", report)
for fold, grid in report["folds"].items():
    print(f"\nFold: {fold} held out")
    for row_name, cells in grid.items():
        print(f"  {row_name}")
        for suite, value in cells.items():
            print(f"      {suite:22s} {value}")

In [ ]:
if FOLD_RESULTS and ready("heads"):
    import matplotlib.pyplot as plt

    # Per-image predictions (private) and the aggregate per-generator table (aggregate, may be committed).
    frames, per_generator = [], []
    for fold in FOLDS:
        for suite, frame in FOLD_EXTRAS[fold]["logreg"]["frames"].items():
            frames.append(frame.assign(fold=fold, suite=suite))
        first = FOLD_RESULTS[fold]["logreg"]["first_seed"]
        for suite in ("known", "unseen"):
            for generator, entry in first.get("suites", {}).get(suite, {}).get("per_generator", {}).items():
                rate = entry.get("detection_rate") or entry.get("false_positive_rate")
                per_generator.append({"fold": fold, "suite": suite, "generator": generator, "kind": entry["kind"], "n": entry["n"],
                                      "flagged_rate": rate["value"], "flagged_rate_ci_low": rate["ci_low"], "flagged_rate_ci_high": rate["ci_high"],
                                      "roc_auc_vs_all_reals": (entry.get("roc_auc_vs_all_reals") or {}).get("value")})
    if frames:
        pd.concat(frames, ignore_index=True).to_parquet(PRIVATE_DIR / "predictions_v1.parquet", index=False)
    pd.DataFrame(per_generator).to_csv(OUTPUT_DIR / "per_generator_v1.csv", index=False)

    # Reliability diagrams: known and unseen, per fold (logistic regression, first seed, after temperature scaling).
    fig, axes = plt.subplots(1, len(FOLDS), figsize=(3.2 * len(FOLDS), 3.2), squeeze=False)
    for ax, fold in zip(axes[0], FOLDS):
        for suite, style in (("known", "o-"), ("unseen", "s--")):
            frame = FOLD_EXTRAS[fold]["logreg"]["frames"].get(suite)
            if frame is not None:
                curve = reliability_curve(frame["prob"].to_numpy(), frame["is_fake"].to_numpy())
                ax.plot([c["mean_predicted"] for c in curve], [c["fraction_fake"] for c in curve], style, markersize=3, label=suite)
        ax.plot([0, 1], [0, 1], "k-", linewidth=0.5, alpha=0.4)
        ax.set_title(f"held out: {fold}", fontsize=8)
        ax.set_xlabel("predicted P(fake)", fontsize=7)
        ax.set_ylabel("observed share of fakes", fontsize=7)
        ax.legend(fontsize=6)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "reliability_v1.png", dpi=130)
    plt.show()

    # DET curves on the unseen test of each fold.
    plt.figure(figsize=(6.5, 5.5))
    for fold in FOLDS:
        frame = FOLD_EXTRAS[fold]["logreg"]["frames"].get("unseen")
        if frame is not None and frame["is_fake"].nunique() == 2:
            det = det_curve_points(frame["prob"].to_numpy(), frame["is_fake"].to_numpy(), positive_label=1)
            plt.plot(np.array(det["far"]) * 100, np.array(det["frr"]) * 100,
                     label=f"{fold} (AUC {metric_value(FOLD_RESULTS[fold]['logreg']['first_seed'], 'unseen'):.3f})")
    plt.plot([0.01, 100], [0.01, 100], "k-", linewidth=0.5, alpha=0.4)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Real images flagged as fake (%)")
    plt.ylabel("Fake images missed (%)")
    plt.title("V1 logistic regression: DET curve on each fold's unseen family")
    plt.legend(fontsize=6, loc="lower left")
    plt.grid(True, which="both", linewidth=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "det_curves_v1.png", dpi=130)
    plt.show()

In [ ]:
# --- Closing paragraph, generated from this run's numbers ---------------------------------------------
parts = []
if not FOLD_RESULTS:
    parts.append(f"Visual Engine V1: stage '{STAGE}' ran; no model has been trained in this run, so every V1 number is not yet run."
                 + (f" Stopped: {STOPPED}." if STOPPED else ""))
else:
    lr = {f: FOLD_RESULTS[f]["logreg"] for f in FOLDS}
    gaps = {f: r["first_seed"].get("known_minus_unseen", {}).get("roc_auc") for f, r in lr.items()}
    gap_text = "; ".join(f"{f}: known {cell(lr[f], 'known')}, unseen {cell(lr[f], 'unseen')}, gap {g['difference']:+.4f} "
                         f"[{g['ci_low']:+.4f}, {g['ci_high']:+.4f}]" for f, g in gaps.items() if g)
    parts.append(f"V1 logistic regression on frozen CLIP embeddings, leave-one-family-out over {len(FOLDS)} families (ROC-AUC, 95% bootstrap CI, "
                 f"first seed). Known-minus-unseen gap per fold -- {gap_text}.")
    real_gaps = [f for f, g in gaps.items() if g and g["excludes_zero"] and g["difference"] > 0]
    values = [g["difference"] for g in gaps.values() if g]
    if values:
        parts.append(f"The gap's CI excludes zero for {len(real_gaps)} of {len(values)} folds ({', '.join(real_gaps) or 'none'}); "
                     f"across folds it runs from {min(values):+.4f} to {max(values):+.4f}. Folds share training data, so they are "
                     "shown one by one, not pooled.")
    unseen_auc = {f: metric_value(lr[f]["first_seed"], "unseen") for f in FOLDS}
    unseen_auc = {f: v for f, v in unseen_auc.items() if v is not None}
    if unseen_auc:
        worst_family = min(unseen_auc, key=unseen_auc.get)
        parts.append(f"The family V1 generalizes to worst is {worst_family} (unseen ROC-AUC {cell(lr[worst_family], 'unseen')}).")
    verdicts_seen = Counter(v["verdict"] for f in FOLDS for v in FOLD_RESULTS[f].get("mlp_minus_logreg", {}).values())
    parts.append(f"MLP against logistic regression on the same test images: {dict(verdicts_seen)} over the folds' known and unseen "
                 "tests ('tie' = the CI of the difference includes zero).")
    ece = [(f, lr[f]["first_seed"]["suites"]["unseen"]["ece_before_calibration"], lr[f]["first_seed"]["suites"]["unseen"]["metrics"]["ece"]["value"])
           for f in FOLDS if "unseen" in lr[f]["first_seed"].get("suites", {})]
    if ece:
        better = sum(1 for _, before, after in ece if after < before)
        parts.append("Temperature scaling was fitted on seen generators only; on the unseen families ECE went " +
                     "; ".join(f"{f} {before:.4f} -> {after:.4f}" for f, before, after in ece) +
                     f" (lower in {better} of {len(ece)} folds).")
    parts.append(f"Seeds: logistic regression -- {lr[FOLDS[0]]['seed_changes']}; MLP -- {FOLD_RESULTS[FOLDS[0]]['mlp']['seed_changes']}.")
    if OLD_ONLY_RESULT:
        parts.append("Old-only ablation (trained on " + ", ".join(OLD_FAMILIES) + " plus real images), scored on each modern family's "
                     "unseen test: " + "; ".join(f"{f} {v}" for f, v in report["old_only_ablation"].items()) + ".")
    if SHORTCUT is not None and not SHORTCUT["gate"]["passed"]:
        parts.append("Caveat carried by every number above: after the shared pipeline a file-property or source-only baseline still "
                     "separates the classes (" + "; ".join(SHORTCUT["gate"]["failing"]) + ").")
    if FINAL_RESULT:
        parts.append(FINAL_RESULT["what_it_cannot_claim"])
closing_paragraph = " ".join(parts)
print(closing_paragraph)
save_results("closing_paragraph", closing_paragraph)
save_results("run_hours", round(elapsed_hours(), 2))
print(f"\nresults: {RESULTS_PATH} | preflight: {PREFLIGHT_PATH} | stage: {STAGE} | elapsed {elapsed_hours():.2f} h"
      + (f" | halted: {STOPPED}" if STOPPED else ""))